# Cambrian-P on VSI-590K: failure review (training-data diagnostic)

Runs **Cambrian-P-7B** (128 frames) on a frozen **10% sample** of VSI-590K's video questions, scores every answer, and exports **20 distinct videos with verified wrong answers** (10 minimum), each with the question, the model's and the reference answer, why it counts as wrong, the video file and the exact input frames, plus a review interface.

**What this is, and isn't.** VSI-590K is part of the data Cambrian-P was trained on. This is a diagnostic of the model on its own training distribution, **not VSI-Bench** and not a held-out accuracy. The exported cases are chosen *because* the model was wrong, so they say nothing about how often it is wrong.

**Verified facts about the data** (checked at revision `346fbd4e41de…`, 7 Nov 2026), which the code relies on and re-checks:
- One annotation file, `vsi_590k.jsonl` (the guide's `vsi-590k.jsonl` name is outdated): 590,667 records. 374,148 are video records (ADT, ARKitScenes, ProcTHOR, S3DIS, ScanNet, ScanNet++ v2), **all single-turn**; 216,519 are image records (Hypersim, robotics, YouTube room tours), outside this review's population.
- Fields: `conversations` (`human`/`gpt`), `question_type`, `video` or `image`; ProcTHOR records also carry an `id`.
- Media: nine `.tar.gz` archives (236 GB). Each video is stored at its annotation path inside `<source>.tar.gz`. Gzip archives can't be read from the middle, so getting selected videos means streaming through the archive.

**Scale of the full run.** The 10% sample is 37,415 questions on about 4,475 videos (75% of all videos), so it needs all six video archives (172 GB). At very roughly 2–4 s per question on an A100, that's on the order of 20–40 GPU-hours, over many Colab sessions. Everything resumes: rerun the run cell and it continues where it stopped.

**Run order**
1. Section 1 (settings, helper code, configuration check) and section 2 (environment and model).
2. Sections 3–5: discover the data, freeze the sample, plan the media.
3. Section 6 with `RUN_MODE = "pilot"` (default): 2 videos, at most 10 questions. Then the resume check (6b).
4. Set `RUN_MODE = "full"`, rerun section 1's settings cell and section 6, as many times as needed.

**New sessions don't download again.** The environment, model weights, annotation file and dataset card are saved once under `CACHE_ROOT` on Drive (about 25–30 GB, mostly the environment and weights) and restored from there in later sessions; a population summary lets later sessions skip re-parsing the annotations. Video archives are cached too if you set `CACHE_ARCHIVES = True` (needs up to 172 GB of Drive). In a new session, run sections 1–6a top to bottom as usual; each step uses the cache when it can.
5. Sections 7–9: score, pick and export failure cases, review them, write the report.

## 1. Settings and output locations

In [ ]:
# @title 1a. Settings
DATASET_REPO = "nyu-visionx/vsi-590k"
DATASET_REVISION = ""       # Resolve once, persist the SHA, then reuse it.
MODEL_REPO = "nyu-visionx/Cambrian-P-7B"
MODEL_REVISION = ""         # Resolve once and persist.
CODE_COMMIT = ""            # Resolve/pin during setup and persist.
SUBSET_FRACTION = 0.10
SEED = 42
NUM_FRAMES = 128
RUN_MODE = "pilot"          # pilot or full
TARGET_FAILURE_VIDEOS = 20
MIN_FAILURE_VIDEOS = 10
RUN_ID = "vsi590k-10pct-seed42-cambrianp7b-v1"
LOCAL_ROOT = "/content/vsi590k_work"
OUTPUT_ROOT = "/content/drive/MyDrive/Cambrian-P/vsi590k_runs/" + RUN_ID

# Further settings (defaults are fine).
POPULATION = "video"        # the annotation population sampled: VSI-590K's video records (all single-turn)
PILOT_VIDEOS, PILOT_MAX_QA = 2, 10
MAX_RETRIES = 2             # bounded retries of a question after an execution error (same settings)
MEDIA_BATCH_GB = 25         # most extracted video data on local disk at once
FAILURE_POOL_VIDEOS = 60    # wrong-answer videos copied to Drive as they are found (avoids re-streaming archives)
REPO_URL = "https://github.com/cambrian-mllm/cambrian-p.git"
# Generation: the model wrapper's defaults (lmms-eval cambrian_p._run_inference_step), written out so they are pinned.
GENERATION = {"max_new_tokens": 1024, "temperature": 0, "do_sample": False, "top_p": None, "num_beams": 1}
PREPROCESSING = ("frames = np.linspace(0, total-1, NUM_FRAMES, dtype=int) decoded with decord (PyAV fallback, recorded); "
                 "expand2square with the image-processor mean colour; SigLIP image processor; fp16")
PROMPT_POLICY = "human-turn-verbatim-minus-leading-image-token-v1"   # must equal vsi590k_lib.PROMPT_POLICY

# Drive cache shared by every run and every new Colab session, so nothing is downloaded or installed twice.
CACHE_ROOT = "/content/drive/MyDrive/Cambrian-P/cache"
CACHE_ENV = True            # keep the installed environment on Drive (~8 GB); later sessions restore it in minutes
CACHE_MODEL = True          # keep the model weights on Drive; later sessions copy them instead of downloading
CACHE_ARCHIVES = False      # also keep video archives on Drive (the full run reads 172 GB of them; needs that much Drive space)

import os, json, shutil, subprocess, sys
assert RUN_MODE in ("pilot", "full"), "RUN_MODE must be 'pilot' or 'full'"
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)
os.makedirs(CACHE_ROOT, exist_ok=True)
os.makedirs("/content/vsi590k", exist_ok=True)     # the helper files below are written here (%%writefile needs the folder)

# Optional Hugging Face token from Colab secrets (key icon, name HF_TOKEN). VSI-590K and the model are public, so
# none is needed; it only raises rate limits. Never printed.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or None
except Exception:
    pass

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
du = shutil.disk_usage("/content")
print(f"Run mode:      {RUN_MODE}")
print(f"Drive output:  {OUTPUT_ROOT}   (manifests, results, cases: persistent)")
print(f"Local work:    {LOCAL_ROOT}   (working copies, lost when the runtime ends; the Drive cache keeps what later sessions need)")
print(f"GPU:           {gpu or 'none found: inference needs a GPU (Runtime > Change runtime type > A100)'}")
print(f"Local disk:    {du.free / 1e9:.0f} GB free of {du.total / 1e9:.0f} GB")
print(f"Drive cache:   {CACHE_ROOT}   (environment: {CACHE_ENV}, model: {CACHE_MODEL}, archives: {CACHE_ARCHIVES})")
print("Anything not in the cache is downloaded from Hugging Face once and saved there; nothing needs uploading by hand.")

### 1b. Helper code: `vsi590k_lib.py`

In [ ]:
%%writefile /content/vsi590k/vsi590k_lib.py
"""Shared logic for the Cambrian-P / VSI-590K failure review (no GPU, no model code).

Imported by the notebook kernel and by the inference worker. Everything here is deterministic and
unit-tested by the notebook's verification cell.
"""
import hashlib
import html
import json
import math
import os
import random
import re
import shutil
import tarfile
import time

LIB_VERSION = "vsi590k-lib-1"
PARSER_VERSION = "parser-v1"
SCORING_POLICY_VERSION = "scoring-v1"
PROMPT_POLICY = "human-turn-verbatim-minus-leading-image-token-v1"
HISTORY_POLICY = "single_turn"
VERDICTS = ("correct", "wrong", "imperfect", "needs_review", "format_error", "unscorable", "execution_error")
EVIDENCE_JUDGMENTS = ("sufficient_in_input", "partial_in_input", "found_only_in_extra_frames",
                      "not_found_in_reviewed_frames", "ambiguous")
FAILURE_CATEGORIES = ("perception_missed_object", "perception_misread_text_or_attribute", "counting_error",
                      "metric_estimation_error", "spatial_relation_error", "temporal_order_error",
                      "orientation_or_viewpoint_error", "possible_annotation_error", "other")
VIDEO_SOURCES = ("adt", "arkitscenes", "procthor", "s3dis", "scannet", "scannetppv2")
OPTION_RE = re.compile(r"^([A-H])\. (.*)$", re.M)
NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?")


# ----------------------------------------------------------------------------------------------- hashing / ids
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def stable_hash(obj, n=None):
    s = hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    return s[:n] if n else s


def record_id(dataset_revision, source_file, line_no):
    """Stable id of one annotation record: dataset revision + annotation file + 1-based line number."""
    return f"vsi590k@{dataset_revision[:12]}/{source_file}#L{line_no}"


def qa_id(rec_id, turn_index):
    return "qa_" + hashlib.sha1(f"{rec_id}|t{turn_index}".encode()).hexdigest()[:20]


def safe_key(video_key):
    """Filesystem-safe, collision-proof name for a media path; the mapping is kept in the manifests."""
    base = re.sub(r"[^A-Za-z0-9._-]+", "__", os.path.splitext(video_key)[0]).strip("_")[:80]
    return f"{base}-{hashlib.sha1(video_key.encode()).hexdigest()[:8]}"


def config_identity(settings):
    """Hash of every setting that changes model inputs or outputs. Run mode is deliberately excluded."""
    keys = ("DATASET_REPO", "DATASET_REVISION", "MODEL_REPO", "MODEL_REVISION", "CODE_COMMIT", "SUBSET_FRACTION",
            "SEED", "NUM_FRAMES", "POPULATION", "PROMPT_POLICY", "GENERATION", "PREPROCESSING")
    missing = [k for k in keys if k not in settings]
    if missing:
        raise ValueError(f"config_identity: missing {missing}")
    return stable_hash({k: settings[k] for k in keys}, 16)


def check_run_config(path, identity_settings):
    """Create run_config.json, or verify the existing one. Raises on any change to the run's identity."""
    current = {"identity": identity_settings, "config_hash": config_identity(identity_settings)}
    if os.path.exists(path):
        saved = json.load(open(path))
        if saved.get("config_hash") != current["config_hash"]:
            diff = {k: (saved["identity"].get(k), v) for k, v in identity_settings.items() if saved["identity"].get(k) != v}
            raise RuntimeError("These settings differ from the ones this RUN_ID was started with (saved -> now): "
                               f"{json.dumps(diff, default=str)}. Restore them, or set a new RUN_ID for a new run.")
        return saved
    write_json_atomic(path, current)
    return current


# ----------------------------------------------------------------------------------------------- files
def write_json_atomic(path, obj):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = f"{path}.tmp-{os.getpid()}"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def write_jsonl_atomic(path, rows):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = f"{path}.tmp-{os.getpid()}"
    with open(tmp, "w") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def read_jsonl(path, tolerate_truncated_tail=True):
    """Rows of a JSONL file. A truncated last line (from an interrupted append) is skipped and reported."""
    rows, bad = [], 0
    if not os.path.exists(path):
        return rows, bad
    with open(path) as f:
        lines = f.read().split("\n")
    for k, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            if tolerate_truncated_tail and k >= len(lines) - 2:
                bad += 1
            else:
                raise
    return rows, bad


class Journal:
    """Append-only JSONL with fsync after every row: an interrupt loses at most the row being written."""

    def __init__(self, path):
        self.path = path
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def append(self, row):
        with open(self.path, "a") as f:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
            f.flush()
            os.fsync(f.fileno())


# ----------------------------------------------------------------------------------------------- schema adapter
def check_record_schema(rec):
    """Problems with one raw VSI-590K record, against the schema verified at the pinned revision."""
    problems = []
    conv = rec.get("conversations")
    if not isinstance(conv, list) or not conv:
        return ["no conversations"]
    if len(conv) % 2:
        problems.append("odd number of turns")
    for k, turn in enumerate(conv):
        if turn.get("from") != ("human" if k % 2 == 0 else "gpt"):
            problems.append(f"turn {k} is from {turn.get('from')!r}")
        if not isinstance(turn.get("value"), str):
            problems.append(f"turn {k} has no text")
    if ("video" in rec) == ("image" in rec):
        problems.append("needs exactly one of video/image")
    if not isinstance(rec.get("question_type"), str):
        problems.append("no question_type")
    return problems


def parse_options(prompt):
    return {letter: text.strip() for letter, text in OPTION_RE.findall(prompt)}


def strip_framing(prompt):
    """Question sentence(s) without the fixed framing and option lines (for display only)."""
    keep = [l for l in prompt.splitlines() if l.strip() and l.strip() != "These are frames of a video."
            and not l.startswith(("Please answer the question", "Answer with the option", "Options:"))
            and not OPTION_RE.match(l)]
    return " ".join(keep)


def answer_format(prompt, reference):
    if parse_options(prompt):
        return "mc_letter"
    if NUMBER_RE.fullmatch(reference.strip()):
        return "numeric"
    return "closed_text"


def normalize_record(rec, line_no, source_file, dataset_revision, population="video"):
    """One raw record -> (qa_rows, exclusion or None). Video records in VSI-590K are single-turn."""
    rid = record_id(dataset_revision, source_file, line_no)
    media_key = "video" if population == "video" else None
    if media_key not in rec:
        return [], {"record_id": rid, "reason": "outside_population", "detail": "image record"}
    problems = check_record_schema(rec)
    if problems:
        return [], {"record_id": rid, "reason": "schema_problem", "detail": "; ".join(problems)}
    conv = rec["conversations"]
    if len(conv) != 2:
        # Never truncated silently: multi-turn video records would need a history policy, and none exist at the
        # pinned revision. If one appears, it is excluded and counted.
        return [], {"record_id": rid, "reason": "multi_turn_unsupported", "detail": f"{len(conv) // 2} QA turns"}
    human, ref = conv[0]["value"], conv[1]["value"].strip()
    if human.count("<image>") != 1 or not human.startswith("<image>"):
        return [], {"record_id": rid, "reason": "unexpected_media_token", "detail": f"{human.count('<image>')} tokens"}
    if not ref:
        return [], {"record_id": rid, "reason": "missing_reference", "detail": ""}
    prompt = human[len("<image>"):].lstrip("\n")
    video_key = rec["video"]
    row = {
        "qa_id": qa_id(rid, 0), "record_id": rid, "native_id": rec.get("id"), "source_file": source_file,
        "source_line": line_no, "turn_index": 0, "source_dataset": video_key.split("/")[0], "video_key": video_key,
        "safe_key": safe_key(video_key), "original_media_path": video_key, "question": strip_framing(prompt),
        "prompt": prompt, "reference_answer": ref, "options": parse_options(prompt),
        "question_type": rec["question_type"], "answer_format": answer_format(prompt, ref),
        "history_policy": HISTORY_POLICY,
    }
    return [row], None


def prompt_excludes_reference(row, raw_record):
    """True if the prompt is exactly the first human turn minus its leading media token, so no assistant turn
    (the reference answer) can be in it."""
    human = raw_record["conversations"][0]["value"]
    return row["prompt"] == human[len("<image>"):].lstrip("\n") and len(raw_record["conversations"]) == 2


# ----------------------------------------------------------------------------------------------- selection
def select_indices(population_size, fraction, seed):
    """Uniform sample without replacement of ceil(fraction * N) positions; sorted for stable files."""
    k = math.ceil(fraction * population_size)
    return sorted(random.Random(seed).sample(range(population_size), k))


def group_by_video(qa_rows):
    groups = {}
    for r in qa_rows:
        groups.setdefault(r["video_key"], []).append(r)
    return groups


def pilot_subset(qa_rows, archive_sizes, n_videos=2, max_qa=10):
    """Deterministic pilot: the videos in the smallest needed archive (cheapest to fetch), first by hash."""
    groups = group_by_video(qa_rows)
    order = sorted(groups, key=lambda v: (archive_sizes.get(archive_for(v)[0], float("inf")),
                                          hashlib.sha1(v.encode()).hexdigest()))
    chosen, out = order[:n_videos], []
    per_video = max(1, max_qa // max(1, len(chosen)))
    for v in chosen:
        out += sorted(groups[v], key=lambda r: r["qa_id"])[:per_video]
    return out[:max_qa]


# ----------------------------------------------------------------------------------------------- media
def archive_for(video_key):
    """VSI-590K stores every video at its annotation path inside <first folder>.tar.gz (verified)."""
    source = video_key.split("/")[0]
    return f"{source}.tar.gz", video_key


def safe_extract_path(dest_root, member_name):
    """Destination for a tar member, refusing absolute paths, '..' and anything outside dest_root."""
    if not member_name or member_name.startswith(("/", "\\")) or re.match(r"^[A-Za-z]:", member_name):
        raise ValueError(f"unsafe member path {member_name!r}")
    norm = os.path.normpath(member_name)
    if norm.startswith("..") or os.path.isabs(norm) or "/../" in f"/{norm}/":
        raise ValueError(f"unsafe member path {member_name!r}")
    dest = os.path.realpath(os.path.join(dest_root, norm))
    root = os.path.realpath(dest_root)
    if not dest.startswith(root + os.sep):
        raise ValueError(f"member escapes the destination: {member_name!r}")
    return dest


def extract_members(fileobj, wanted, dest_root, byte_budget=None, log=print):
    """Stream a .tar.gz once and write only the wanted regular-file members under dest_root.

    Stops as soon as every wanted member is written, or before starting a member that would exceed byte_budget.
    Returns {"written": {member: path}, "bytes": n, "stopped": reason, "scanned": members_seen}.
    """
    wanted = set(wanted)
    written, total, scanned = {}, 0, 0
    stopped = "end_of_archive"
    with tarfile.open(fileobj=fileobj, mode="r|gz") as tf:
        for m in tf:
            scanned += 1
            if m.name not in wanted or m.name in written:
                continue
            if not m.isreg():
                log(f"  skipped {m.name}: not a regular file")
                continue
            if byte_budget is not None and written and total + m.size > byte_budget:
                stopped = "byte_budget"
                break
            dest = safe_extract_path(dest_root, m.name)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            src = tf.extractfile(m)
            tmp = dest + ".part"
            with open(tmp, "wb") as out:
                shutil.copyfileobj(src, out, 1 << 20)
            if os.path.getsize(tmp) != m.size:
                os.remove(tmp)
                raise IOError(f"{m.name}: wrote {os.path.getsize(tmp)} bytes, expected {m.size}")
            os.replace(tmp, dest)
            written[m.name] = dest
            total += m.size
            if len(written) == len(wanted):
                stopped = "all_found"
                break
    return {"written": written, "bytes": total, "stopped": stopped, "scanned": scanned}


# ----------------------------------------------------------------------------------------------- shards
def write_shard(shard_dir, name, rows, config_hash):
    """Durable shard: rows file plus a completion marker holding its hash. Written atomically, marker last."""
    os.makedirs(shard_dir, exist_ok=True)
    data = os.path.join(shard_dir, f"{name}.jsonl")
    write_jsonl_atomic(data, rows)
    marker = {"name": name, "rows": len(rows), "sha256": sha256_file(data), "config_hash": config_hash,
              "finished": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "lib": LIB_VERSION}
    write_json_atomic(os.path.join(shard_dir, f"{name}.done.json"), marker)
    return data


def load_shards(shard_dir, config_hash):
    """Rows of every complete, verified shard; incomplete, corrupted or foreign shards are reported, not used."""
    rows, report = [], {"complete": 0, "incomplete": [], "corrupted": [], "other_config": []}
    if not os.path.isdir(shard_dir):
        return rows, report
    for fn in sorted(os.listdir(shard_dir)):
        if not fn.endswith(".jsonl") or ".tmp-" in fn:
            continue
        name = fn[:-len(".jsonl")]
        marker_path = os.path.join(shard_dir, f"{name}.done.json")
        if not os.path.exists(marker_path):
            report["incomplete"].append(name)
            continue
        marker = json.load(open(marker_path))
        if marker.get("config_hash") != config_hash:
            report["other_config"].append(name)
            continue
        if sha256_file(os.path.join(shard_dir, fn)) != marker.get("sha256"):
            report["corrupted"].append(name)
            continue
        part, _ = read_jsonl(os.path.join(shard_dir, fn), tolerate_truncated_tail=False)
        rows += part
        report["complete"] += 1
    return rows, report


def merge_results(rows, config_hash):
    """One outcome per qa_id: the first successful answer wins; failed attempts are kept separately."""
    ok, failed, video_meta = {}, {}, {}
    for r in rows:
        if r.get("config_hash") != config_hash:
            continue
        if r.get("kind") == "video_meta":
            video_meta.setdefault(r["video_key"], r)
            continue
        if r.get("status") == "ok":
            # Deterministic whatever the file order: the earliest recorded answer wins.
            prev = ok.get(r["qa_id"])
            if prev is None or (r.get("answered_at") or "") < (prev.get("answered_at") or ""):
                ok[r["qa_id"]] = r
        else:
            failed.setdefault(r["qa_id"], []).append(r)
    for q in list(failed):
        if q in ok:
            del failed[q]
    return ok, failed, video_meta


# ----------------------------------------------------------------------------------------------- scoring
def norm_text(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s,\-']", " ", s.lower())).strip()


def parse_mc(answer, options):
    """Documented rule: a leading option letter (optionally in parentheses, followed by '.', ':', ')' or a space),
    or else a reply that exactly equals one option's text (case and punctuation ignored)."""
    a = answer.strip()
    m = re.match(r"^\(?([A-H])\)?(?:[.:)\s]|$)", a)
    if m and m.group(1) in options:
        return m.group(1), "leading_letter"
    hits = [k for k, v in options.items() if norm_text(v) == norm_text(a)]
    if len(hits) == 1:
        return hits[0], "option_text"
    return None, "unparseable"


def mean_relative_accuracy(pred, target):
    """VSI-Bench's MRA (lmms-eval/lmms_eval/tasks/vsibench/utils.py): share of thresholds 0.50..0.95 met."""
    if target == 0:
        return 1.0 if pred == 0 else 0.0
    rel = abs(pred - target) / abs(target)
    thresholds = [0.5 + 0.05 * k for k in range(10)]
    return sum(rel <= 1 - t + 1e-12 for t in thresholds) / 10


def angular_error(pred, target):
    d = abs(pred - target) % 360
    return min(d, 360 - d)


ANGLE_TOLERANCE = {"correct_max_deg": 15.0, "wrong_min_deg": 45.0}
DIRECTION_VOCAB = ("front-left", "front-right", "back-left", "back-right", "left", "right", "back", "front")


def score_answer(row, raw_answer, status="ok"):
    """Verdict for one answer. Only 'wrong' counts as a verified failure."""
    base = {"qa_id": row["qa_id"], "parser_version": PARSER_VERSION, "policy": SCORING_POLICY_VERSION,
            "raw_answer": raw_answer, "reference": row.get("reference_answer", "")}
    if status != "ok":
        return dict(base, verdict="execution_error", score=None, metric=None, parsed=None, basis="inference did not complete")
    ref = (row.get("reference_answer") or "").strip()
    if not ref:
        return dict(base, verdict="unscorable", score=None, metric=None, parsed=None, basis="no reference answer")
    fmt, ans = row["answer_format"], (raw_answer or "").strip()
    if fmt == "mc_letter":
        options = row["options"]
        ref_letter = ref[0] if re.match(r"^[A-H](\.|$)", ref) else None
        if ref_letter not in options:
            return dict(base, verdict="unscorable", score=None, metric="mc_exact", parsed=None,
                        basis=f"reference {ref!r} is not one of the options")
        got, how = parse_mc(ans, options)
        if got is None:
            return dict(base, verdict="format_error", score=None, metric="mc_exact", parsed=None,
                        basis="reply is not a valid option letter or option text")
        ok = got == ref_letter
        return dict(base, verdict="correct" if ok else "wrong", score=float(ok), metric="mc_exact", parsed=got,
                    basis=f"parsed {got} ({how}) vs reference {ref_letter} ({options.get(ref_letter, '')!r})")
    if fmt == "numeric":
        target = float(ref)
        nums = NUMBER_RE.findall(ans)
        if not nums:
            return dict(base, verdict="format_error", score=None, metric=None, parsed=None, basis="no number in reply")
        pred = float(nums[0])
        if row["question_type"] == "absolute_direction_object":
            err = angular_error(pred, target)
            v = ("correct" if err <= ANGLE_TOLERANCE["correct_max_deg"] else
                 "wrong" if err > ANGLE_TOLERANCE["wrong_min_deg"] else "imperfect")
            return dict(base, verdict=v, score=round(1 - err / 180, 4), metric="angular_error_diagnostic", parsed=pred,
                        basis=f"angular error {err:.1f} deg (correct <= {ANGLE_TOLERANCE['correct_max_deg']:g}, "
                              f"wrong > {ANGLE_TOLERANCE['wrong_min_deg']:g}; diagnostic thresholds, not an official metric)")
        mra = mean_relative_accuracy(pred, target)
        v = "correct" if mra == 1.0 else "wrong" if mra == 0.0 else "imperfect"
        rel = abs(pred - target) / abs(target) if target else float("inf")
        return dict(base, verdict=v, score=mra, metric="mra_diagnostic", parsed=pred,
                    basis=f"{pred:g} vs {target:g} (relative error {rel:.0%}); VSI-Bench-style MRA used as a diagnostic: "
                          "1.0 = within 5% (correct), 0 = more than 50% off (wrong), else imperfect")
    # closed_text: ProcTHOR questions answered in words, from a closed set given in the question.
    qtype, nref, nans = row["question_type"], norm_text(ref), norm_text(ans)
    if qtype == "relative_direction_object":
        hits = [w for w in DIRECTION_VOCAB if re.search(rf"(?<![\w-]){re.escape(w)}(?![\w-])", nans)]
        if len(hits) == 1:
            ok = hits[0] == nref
            return dict(base, verdict="correct" if ok else "wrong", score=float(ok), metric="closed_vocab_exact",
                        parsed=hits[0], basis=f"direction {hits[0]!r} vs reference {nref!r} (closed vocabulary)")
        return dict(base, verdict="needs_review", score=None, metric="closed_vocab_exact", parsed=None,
                    basis="reply does not name exactly one direction")
    m = re.search(r"\(([^)]*)\)", row["prompt"]) if qtype == "relative_distance_object" else \
        re.search(r"categories in the video: ([^?\n]*)\?", row["prompt"]) if qtype == "appearance_order" else None
    candidates = [norm_text(c) for c in m.group(1).split(",")] if m else []
    if qtype == "relative_distance_object" and candidates:
        if nans == nref:
            return dict(base, verdict="correct", score=1.0, metric="closed_set_exact", parsed=nans, basis="matches reference")
        if nans in candidates:
            return dict(base, verdict="wrong", score=0.0, metric="closed_set_exact", parsed=nans,
                        basis=f"named a different listed object ({nans!r} vs {nref!r})")
        return dict(base, verdict="needs_review", score=None, metric="closed_set_exact", parsed=None,
                    basis="reply is not one of the listed objects; may be a synonym")
    if qtype == "appearance_order" and candidates:
        got = [norm_text(x) for x in ans.split(",") if x.strip()]
        want = [norm_text(x) for x in ref.split(",") if x.strip()]
        if got == want:
            return dict(base, verdict="correct", score=1.0, metric="sequence_exact", parsed=got, basis="same order")
        if sorted(got) == sorted(want):
            return dict(base, verdict="wrong", score=0.0, metric="sequence_exact", parsed=got,
                        basis="a valid ordering of the listed categories, but not the reference order")
        return dict(base, verdict="needs_review", score=None, metric="sequence_exact", parsed=None,
                    basis="reply is not an ordering of the listed categories")
    return dict(base, verdict="needs_review", score=None, metric=None, parsed=None,
                basis="free-form reply with no verified rule; compare meaning by hand")


def apply_review(score_row, review):
    """A reviewer's final decision overrides the automatic verdict and is recorded as such."""
    if not review or not review.get("final_verdict"):
        return score_row
    return dict(score_row, verdict=review["final_verdict"], verdict_basis_auto=score_row["basis"],
                basis=f"reviewer: {review.get('final_notes', '').strip() or 'no note'}", reviewed=True)


# ----------------------------------------------------------------------------------------------- failure selection
def select_failure_cases(scores, qa_by_id, target=20):
    """Distinct videos with a verified wrong answer, ordered for diversity across sources and question types.

    Deterministic: ties broken by hash. Returns (selected, reserve), each a list of {video_key, qa_id, ...}.
    """
    by_video = {}
    for s in scores:
        if s["verdict"] == "wrong":
            q = qa_by_id[s["qa_id"]]
            by_video.setdefault(q["video_key"], []).append(q)
    h = lambda x: hashlib.sha1(x.encode()).hexdigest()
    remaining = dict(by_video)
    src_n, type_n, order = {}, {}, []
    while remaining:
        def key(v):
            qs = remaining[v]
            return (src_n.get(qs[0]["source_dataset"], 0), min(type_n.get(q["question_type"], 0) for q in qs), h(v))
        v = min(remaining, key=key)
        qs = remaining.pop(v)
        primary = min(qs, key=lambda q: (type_n.get(q["question_type"], 0), h(q["qa_id"])))
        src_n[primary["source_dataset"]] = src_n.get(primary["source_dataset"], 0) + 1
        type_n[primary["question_type"]] = type_n.get(primary["question_type"], 0) + 1
        order.append({"video_key": v, "safe_key": primary["safe_key"], "qa_id": primary["qa_id"],
                      "source_dataset": primary["source_dataset"], "question_type": primary["question_type"],
                      "other_wrong_qa_ids": sorted(q["qa_id"] for q in qs if q["qa_id"] != primary["qa_id"])})
    return order[:target], order[target:]


def shortfall_message(n_selected, target, minimum):
    if n_selected >= target:
        return f"Target met: {n_selected} distinct videos with verified wrong answers."
    if n_selected >= minimum:
        return (f"Shortfall: {n_selected} distinct videos with verified wrong answers (target {target}, minimum "
                f"{minimum} met).")
    return (f"Minimum NOT met: only {n_selected} distinct videos with verified wrong answers in the processed part of "
            f"the frozen selection (minimum {minimum}, target {target}). The selection was not expanded.")


# ----------------------------------------------------------------------------------------------- frames
def sample_indices(total_frames, num_frames):
    """The model wrapper's sampling (lmms-eval cambrian_p.load_video): np.linspace(0, total-1, k, dtype=int)."""
    import numpy as np
    if total_frames <= 0:
        raise ValueError("video has no frames")
    return [int(i) for i in np.linspace(0, total_frames - 1, num_frames, dtype=int)]


def frame_name(position, frame_index, seconds):
    """Input-frame file name keeping position, frame index and timestamp as separate fields."""
    return f"in{position:03d}_frame{frame_index:06d}_t{seconds:09.3f}s.jpg"


def parse_frame_name(name):
    m = re.fullmatch(r"(in|x)(\d{3})_frame(\d{6})_t(\d+\.\d{3})s\.jpg", name)
    if not m:
        return None
    return {"set": "input" if m.group(1) == "in" else "extra", "position": int(m.group(2)),
            "frame_index": int(m.group(3)), "seconds": float(m.group(4))}


def check_case_images(case_dir, provenance):
    """Problems with a case's evidence images: unknown names, or input frames not in the recorded sample."""
    problems = []
    allowed = set(zip(provenance["frame_indices"], [round(t, 3) for t in provenance["timestamps_s"]]))
    for sub, kind in (("frames_input", "input"), ("frames_extra", "extra")):
        d = os.path.join(case_dir, sub)
        for fn in sorted(os.listdir(d)) if os.path.isdir(d) else []:
            if not fn.endswith(".jpg") or fn.startswith("sheet_"):
                continue
            p = parse_frame_name(fn)
            if p is None or p["set"] != kind:
                problems.append(f"{sub}/{fn}: name does not identify a {kind} frame")
            elif kind == "input" and (p["frame_index"], round(p["seconds"], 3)) not in allowed:
                problems.append(f"{sub}/{fn}: not one of this video's recorded input frames")
    return problems


# ----------------------------------------------------------------------------------------------- report
def esc(s):
    return html.escape("" if s is None else str(s), quote=True)


def rel_link(target_path, from_dir):
    return os.path.relpath(target_path, from_dir).replace(os.sep, "/")


def case_markdown(case):
    q, s, r = case["qa"], case["score"], case.get("review") or {}
    opts = "\n".join(f"- {k}. {v}" for k, v in sorted(q["options"].items()))
    lines = [
        f"# {q['video_key']}", "",
        f"- QA id: `{q['qa_id']}`  (record `{q['record_id']}`, line {q['source_line']})",
        f"- Source: {q['source_dataset']}; question type: `{q['question_type']}`", "",
        "## Question", "", q["question"], "", opts if opts else "(no options)", "",
        "## Answers", "",
        f"- Model (raw): `{s['raw_answer']}`", f"- Model (parsed): `{s.get('parsed')}`",
        f"- Reference: `{s['reference']}`",
        f"- Verdict: **{s['verdict']}** (metric `{s.get('metric')}`, score {s.get('score')})",
        f"- Why: {s['basis']}", "",
        "## Model input", "",
        f"- {case['provenance']['num_frames']} frames from {case['provenance']['total_frames']} at "
        f"{case['provenance']['fps']:.3f} fps; reader {case['provenance']['reader']}",
        f"- Preprocessing: {case['provenance']['preprocessing']}",
        "- Contact sheets and full-resolution input frames: `frames_input/`; extra diagnostic frames: `frames_extra/`",
        "", "## Review", "",
        f"- Stage: {r.get('stage', 'not reviewed')}",
        f"- Blind judgment (before reveal): {r.get('provisional_answer', '')!s} / {r.get('provisional_evidence', '')!s}",
        f"- Visual evidence: {r.get('evidence', 'not judged')}",
        f"- Tentative failure category: {r.get('category', 'not judged')} (confidence {r.get('confidence', '-')})",
        f"- Observations (visible): {r.get('observations', '')}",
        f"- Hypotheses (not verified): {r.get('hypotheses', '')}",
        "", "_Training-data diagnostic: this case was selected because the model answered wrongly. It says nothing about "
            "how often the model is wrong._",
    ]
    return "\n".join(lines)


def report_html(cases, root, summary_lines):
    """Self-contained page; every link is relative to root so the folder can be copied as a whole."""
    parts = ["<!DOCTYPE html><html><head><meta charset='utf-8'><title>Cambrian-P VSI-590K failure review</title>",
             "<style>body{font-family:system-ui,sans-serif;max-width:1100px;margin:24px auto;padding:0 16px;color:#111}"
             "section{border-top:1px solid #ddd;padding:16px 0}video{max-width:100%;max-height:360px}"
             "img{max-width:100%}.k{color:#555}code{background:#f3f3f3;padding:1px 4px}</style></head><body>",
             "<h1>Cambrian-P on VSI-590K: failure review</h1>",
             "<p><b>Training-data diagnostic.</b> VSI-590K is part of Cambrian-P's training data. These cases were "
             "selected because the answer was wrong; they are not an estimate of the error rate.</p><ul>"]
    parts += [f"<li>{esc(l)}</li>" for l in summary_lines] + ["</ul>"]
    for c in cases:
        q, s, r = c["qa"], c["score"], c.get("review") or {}
        cdir = os.path.join(root, "cases", c["safe_key"])
        opts = "".join(f"<li>{esc(k)}. {esc(v)}</li>" for k, v in sorted(q["options"].items()))
        sheets = "".join(f"<img src='{esc(rel_link(os.path.join(cdir, 'frames_input', f), root))}' alt='input frames'>"
                         for f in sorted(os.listdir(os.path.join(cdir, "frames_input")))
                         if f.startswith("sheet_")) if os.path.isdir(os.path.join(cdir, "frames_input")) else ""
        parts.append(
            f"<section id='{esc(c['safe_key'])}'><h2>{esc(q['video_key'])}</h2>"
            f"<p class='k'>{esc(q['source_dataset'])} · {esc(q['question_type'])} · <code>{esc(q['qa_id'])}</code></p>"
            f"<video controls preload='metadata' src='{esc(rel_link(c['video_file'], root))}'></video>"
            f"<p><b>Question:</b> {esc(q['question'])}</p><ul>{opts}</ul>"
            f"<p><b>Model:</b> {esc(s['raw_answer'])} → {esc(s.get('parsed'))} &nbsp; <b>Reference:</b> {esc(s['reference'])}"
            f" &nbsp; <b>Verdict:</b> {esc(s['verdict'])}</p><p class='k'>{esc(s['basis'])}</p>"
            f"<p><b>Evidence:</b> {esc(r.get('evidence', 'not judged'))} · <b>Category:</b> "
            f"{esc(r.get('category', 'not judged'))} ({esc(r.get('confidence', '-'))})</p>"
            f"<p><a href='{esc(rel_link(os.path.join(cdir, 'case.md'), root))}'>case.md</a></p>{sheets}</section>")
    parts.append("</body></html>")
    return "".join(parts)

`vsi590k_pipeline.py`

In [ ]:
%%writefile /content/vsi590k/vsi590k_pipeline.py
"""Orchestration for the VSI-590K failure review: discovery, frozen selection, media batches, merge, scoring,
case export and report. Network access and the GPU worker are passed in, so the flow can be tested offline."""
import csv
import hashlib
import json
import os
import time

import vsi590k_lib as L


# ----------------------------------------------------------------------------------------------- 3. discovery
def profile_annotations(paths, dataset_revision, population="video"):
    """Schema profile + normalized QA rows + exclusions for the complete annotation set."""
    prof = {"records": 0, "key_sets": {}, "turns": {}, "media": {}, "question_types": {}, "sources": {}}
    rows, exclusions, raw_by_record = [], [], {}
    for path in paths:
        fname = os.path.basename(path)
        with open(path) as f:
            for line_no, line in enumerate(f, 1):
                r = json.loads(line)
                prof["records"] += 1
                ks = ",".join(sorted(r))
                prof["key_sets"][ks] = prof["key_sets"].get(ks, 0) + 1
                n = len(r.get("conversations", [])) // 2
                prof["turns"][n] = prof["turns"].get(n, 0) + 1
                media = "video" if "video" in r else "image" if "image" in r else "none"
                prof["media"][media] = prof["media"].get(media, 0) + 1
                src = (r.get("video") or r.get("image") or "?").split("/")[0]
                prof["sources"][f"{src}:{media}"] = prof["sources"].get(f"{src}:{media}", 0) + 1
                got, exc = L.normalize_record(r, line_no, fname, dataset_revision, population)
                if exc:
                    exclusions.append(exc)
                for q in got:
                    prof["question_types"][q["question_type"]] = prof["question_types"].get(q["question_type"], 0) + 1
                    rows.append(q)
                    raw_by_record[q["record_id"]] = {"record_id": q["record_id"], "source_file": fname,
                                                     "source_line": line_no, "raw": r}
    return rows, exclusions, raw_by_record, prof


def schema_checks(rows, exclusions, prof, raw_by_record):
    """Problems that must be fixed before selection (an empty list means the adapter is consistent)."""
    problems = []
    if len(rows) + len(exclusions) != prof["records"]:
        problems.append("some records map to neither a QA row nor an exclusion")
    if len({r["qa_id"] for r in rows}) != len(rows):
        problems.append("qa ids are not unique")
    bad = [r["qa_id"] for r in rows if not L.prompt_excludes_reference(r, raw_by_record[r["record_id"]]["raw"])]
    if bad:
        problems.append(f"{len(bad)} prompts not built from the human turn alone")
    selfscore = [r["qa_id"] for r in rows if L.score_answer(r, r["reference_answer"])["verdict"] != "correct"]
    if selfscore:
        problems.append(f"{len(selfscore)} references do not score as correct against themselves (parser gap)")
    unexpected = {e["reason"] for e in exclusions} - {"outside_population"}
    if unexpected:
        problems.append(f"unexpected exclusions: {sorted(unexpected)} (listed in exclusions.jsonl)")
    return problems


def population_summary(rows, exclusions, prof, annotation_hashes, population):
    """What a frozen selection is checked against, so later sessions need not parse every record again."""
    return {"annotation_sha256": annotation_hashes, "lib": L.LIB_VERSION, "parser": L.PARSER_VERSION,
            "population": population, "population_size": len(rows),
            "population_hash": hashlib.sha256("\n".join(r["qa_id"] for r in rows).encode()).hexdigest(),
            "videos": len({r["video_key"] for r in rows}),
            "exclusions": {k: sum(1 for e in exclusions if e["reason"] == k) for k in sorted({e["reason"] for e in exclusions})},
            "profile": prof}


def load_population_summary(path, annotation_hashes, population):
    """The cached summary, only if it was made from these exact annotation files with this code."""
    if not os.path.exists(path):
        return None
    s = json.load(open(path))
    ok = (s.get("annotation_sha256") == annotation_hashes and s.get("lib") == L.LIB_VERSION
          and s.get("parser") == L.PARSER_VERSION and s.get("population") == population)
    return s if ok else None


# ----------------------------------------------------------------------------------------------- 4. selection
def freeze_selection(out_root, rows, raw_by_record, exclusions, settings, annotation_hashes, summary=None):
    """Sample once and persist; on a later run, verify the saved selection instead of sampling again.

    rows may be None on a later run: the saved selection is then checked against a verified population summary."""
    sel_path = os.path.join(out_root, "selection.json")
    if rows is not None:
        population_size = len(rows)
        population_hash = hashlib.sha256("\n".join(r["qa_id"] for r in rows).encode()).hexdigest()
    elif summary is not None and os.path.exists(sel_path):
        population_size, population_hash = summary["population_size"], summary["population_hash"]
    else:
        raise RuntimeError("No saved selection yet: the annotations must be parsed in full to sample.")
    if os.path.exists(sel_path):
        saved = json.load(open(sel_path))
        expect = {"dataset_revision": settings["DATASET_REVISION"], "annotation_sha256": annotation_hashes,
                  "population_size": population_size, "population_hash": population_hash,
                  "fraction": settings["SUBSET_FRACTION"], "seed": settings["SEED"]}
        diff = {k: (saved.get(k), v) for k, v in expect.items() if saved.get(k) != v}
        if diff:
            raise RuntimeError(f"The saved selection does not match the current data/settings: {diff}")
        qa, _ = L.read_jsonl(os.path.join(out_root, "qa_manifest.jsonl"), tolerate_truncated_tail=False)
        recs, _ = L.read_jsonl(os.path.join(out_root, "selected_records.jsonl"), tolerate_truncated_tail=False)
        verify_selection_files(saved, qa, recs)
        return saved, qa, False
    idx = L.select_indices(len(rows), settings["SUBSET_FRACTION"], settings["SEED"])
    chosen = [rows[i] for i in idx]
    rec_ids = sorted({q["record_id"] for q in chosen}, key=lambda x: int(x.rsplit("#L", 1)[1]))
    sel = {"sampling_unit": "annotation record (video records; all are single-turn)",
           "dataset_repo": settings["DATASET_REPO"], "dataset_revision": settings["DATASET_REVISION"],
           "annotation_sha256": annotation_hashes, "population": settings["POPULATION"],
           "population_size": len(rows), "population_hash": population_hash,
           "records_outside_population": sum(1 for e in exclusions if e["reason"] == "outside_population"),
           "records_excluded_in_population": sum(1 for e in exclusions if e["reason"] != "outside_population"),
           "fraction": settings["SUBSET_FRACTION"], "seed": settings["SEED"], "selected_records": len(rec_ids),
           "selected_qa": len(chosen), "selected_videos": len({q["video_key"] for q in chosen}),
           "selected_qa_ids_sha256": hashlib.sha256("\n".join(q["qa_id"] for q in chosen).encode()).hexdigest(),
           "method": "random.Random(seed).sample(range(N), ceil(fraction*N)) over population rows in file order",
           "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    L.write_jsonl_atomic(os.path.join(out_root, "selected_records.jsonl"), [raw_by_record[r] for r in rec_ids])
    L.write_jsonl_atomic(os.path.join(out_root, "qa_manifest.jsonl"), chosen)
    L.write_jsonl_atomic(os.path.join(out_root, "exclusions.jsonl"),
                         [e for e in exclusions if e["reason"] != "outside_population"])
    L.write_json_atomic(sel_path, sel)       # written last: its presence means the selection is complete
    return sel, chosen, True


def verify_selection_files(sel, qa, recs):
    ids = [q["qa_id"] for q in qa]
    problems = []
    if len(ids) != sel["selected_qa"] or len(set(ids)) != len(ids):
        problems.append("qa_manifest count/uniqueness")
    if hashlib.sha256("\n".join(ids).encode()).hexdigest() != sel["selected_qa_ids_sha256"]:
        problems.append("qa ids hash")
    if {r["record_id"] for r in recs} != {q["record_id"] for q in qa} or len(recs) != sel["selected_records"]:
        problems.append("selected_records vs qa_manifest")
    if len({q["video_key"] for q in qa}) != sel["selected_videos"]:
        problems.append("video count")
    if problems:
        raise RuntimeError(f"selection files disagree: {problems}")


# ----------------------------------------------------------------------------------------------- 5. media
def media_manifest(qa, archive_sizes):
    out = {}
    for q in qa:
        a, member = L.archive_for(q["video_key"])
        e = out.setdefault(q["video_key"], {"video_key": q["video_key"], "safe_key": q["safe_key"], "archive": a,
                                            "member": member, "archive_bytes": archive_sizes.get(a), "member_bytes": None,
                                            "n_qa": 0})
        e["n_qa"] += 1
    return sorted(out.values(), key=lambda e: e["video_key"])


def transfer_plan(manifest, archive_sizes):
    need = sorted({e["archive"] for e in manifest}, key=lambda a: archive_sizes.get(a) or float("inf"))
    return [{"archive": a, "archive_bytes": archive_sizes.get(a), "videos": sum(1 for e in manifest if e["archive"] == a),
             "selected_video_bytes": "unknown until extracted"} for a in need]


# ----------------------------------------------------------------------------------------------- 6. results
def gather_results(shard_dir, journal_path, config_hash):
    shard_rows, report = L.load_shards(shard_dir, config_hash)
    journal_rows, truncated = L.read_jsonl(journal_path)
    ok, failed, meta = L.merge_results(shard_rows + journal_rows, config_hash)
    report["journal_truncated_lines"] = truncated
    return ok, failed, meta, report


def run_selection(qa_subset, manifest_by_video, ctx, log=print):
    """Process every pending question in qa_subset, archive by archive, in bounded media batches.

    ctx keys: open_archive(archive, prefer_download) -> file object; run_worker(batch_path, extra_args) -> exit code;
    media_dir, work_dir, shard_dir, journal, config_hash, batch_bytes, archive_sizes, keep_media(fn), log_missing(fn),
    prefer_download (bool).
    """
    ok, failed, _, _ = gather_results(ctx["shard_dir"], ctx["journal"], ctx["config_hash"])
    skip = set(ctx.get("skip_videos", ()))        # media already confirmed missing (retry by clearing the log)
    by_video = L.group_by_video([q for q in qa_subset if q["qa_id"] not in ok and q["video_key"] not in skip])
    if not by_video:
        log("Nothing pending: every question in this set already has a verified answer.")
        return {"pending_videos": 0}
    archives = sorted({manifest_by_video[v]["archive"] for v in by_video},
                      key=lambda a: ctx["archive_sizes"].get(a) or float("inf"))
    summary = {"pending_videos": len(by_video), "processed_videos": 0, "missing": []}
    for archive in archives:
        wanted = {manifest_by_video[v]["member"]: v for v in by_video if manifest_by_video[v]["archive"] == archive}
        log(f"\n=== {archive}: {len(wanted)} video(s) pending ===")
        while wanted:
            fobj = ctx["open_archive"](archive, ctx.get("prefer_download", False))
            try:
                out = L.extract_members(fobj, set(wanted), ctx["media_dir"], ctx["batch_bytes"], log=log)
            finally:
                fobj.close()
            got = out["written"]
            log(f"  extracted {len(got)} video(s), {out['bytes'] / 1e9:.2f} GB ({out['stopped']}, {out['scanned']} members scanned)")
            if not got:
                for member, v in wanted.items():
                    ctx["log_missing"](v, archive, member)
                    summary["missing"].append(v)
                break
            batch = {"videos": [{"video_key": wanted[mb], "safe_key": manifest_by_video[wanted[mb]]["safe_key"],
                                 "local_path": path, "member_bytes": os.path.getsize(path),
                                 "qa": by_video[wanted[mb]]} for mb, path in got.items()]}
            bpath = os.path.join(ctx["work_dir"], f"batch-{int(time.time() * 1000)}.json")
            L.write_json_atomic(bpath, batch)
            code = ctx["run_worker"](bpath, [])
            for mb, path in got.items():
                ctx["keep_media"](wanted[mb], path)       # e.g. copy wrong-answer videos to the Drive pool
                if os.path.exists(path):
                    os.remove(path)
                wanted.pop(mb)
            summary["processed_videos"] += len(got)
            if code != 0:
                log(f"  worker exited with code {code}; stopping so the cause can be checked. Rerun to resume.")
                summary["stopped"] = code
                return summary
            if out["stopped"] == "end_of_archive" and wanted:
                for member, v in wanted.items():
                    ctx["log_missing"](v, archive, member)
                    summary["missing"].append(v)
                break
    return summary


# ----------------------------------------------------------------------------------------------- 7. scoring
def score_all(qa, ok, failed, reviews_by_qa, missing_videos=()):
    scores, missing_videos = [], set(missing_videos)
    for q in qa:
        if q["qa_id"] in ok:
            s = L.score_answer(q, ok[q["qa_id"]]["raw_answer"])
        elif q["video_key"] in missing_videos:
            s = L.score_answer(q, "", status="error")
            s["error"] = "media_missing: video not found in its archive"
        elif q["qa_id"] in failed:
            s = L.score_answer(q, "", status="error")
            s["error"] = failed[q["qa_id"]][-1].get("error")
        else:
            continue
        s.update(video_key=q["video_key"], question_type=q["question_type"], source_dataset=q["source_dataset"])
        scores.append(L.apply_review(s, reviews_by_qa.get(q["qa_id"])))
    return scores


def coverage(sel, qa, ok, failed, scores, missing, mode):
    verdicts = {}
    for s in scores:
        verdicts[s["verdict"]] = verdicts.get(s["verdict"], 0) + 1
    done_v = {q["video_key"] for q in qa if q["qa_id"] in ok}
    rec_done = {q["record_id"] for q in qa if q["qa_id"] in ok}
    by = lambda key: {k: sum(1 for s in scores if s.get(key) == k and s["verdict"] == "wrong") for k in
                      sorted({s.get(key) for s in scores})}
    return {"run_mode_last": mode, "selected": {"records": sel["selected_records"], "qa": sel["selected_qa"],
                                               "videos": sel["selected_videos"], "fraction": sel["fraction"]},
            "completed": {"records": len(rec_done), "qa": len(ok), "videos_with_any_answer": len(done_v)},
            "failed_qa": sum(1 for s in scores if s["verdict"] == "execution_error"),
            "pending_qa": sel["selected_qa"] - len(scores),
            "missing_media_videos": sorted(set(missing)), "verdicts": verdicts,
            "wrong_by_source": by("source_dataset"), "wrong_by_type": by("question_type"),
            "full_run_complete": len(scores) == sel["selected_qa"]}


def write_score_outputs(out_root, ok, failed, scores, qa_by_id):
    L.write_jsonl_atomic(os.path.join(out_root, "predictions.jsonl"), [ok[k] for k in sorted(ok)])
    L.write_jsonl_atomic(os.path.join(out_root, "errors.jsonl"), [r for k in sorted(failed) for r in failed[k]])
    L.write_jsonl_atomic(os.path.join(out_root, "scores.jsonl"), scores)
    cols = ["video_key", "qa_id", "source_dataset", "question_type", "question", "raw_answer", "parsed", "reference",
            "verdict", "metric", "score", "basis"]
    for name, verdict in (("failure_candidates.csv", "wrong"), ("imperfect_cases.csv", "imperfect"),
                          ("needs_review.csv", "needs_review")):
        with open(os.path.join(out_root, name), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
            w.writeheader()
            for s in scores:
                if s["verdict"] == verdict:
                    w.writerow(dict(s, question=qa_by_id[s["qa_id"]]["question"]))


# ----------------------------------------------------------------------------------------------- 8/9. export
def load_reviews(review_dir):
    cases, by_qa = {}, {}
    for sub in ("cases", "qa"):
        d = os.path.join(review_dir, sub)
        for fn in sorted(os.listdir(d)) if os.path.isdir(d) else []:
            if fn.endswith(".json"):
                r = json.load(open(os.path.join(d, fn)))
                if sub == "cases":
                    cases[r["safe_key"]] = r
                if r.get("final_verdict") and r.get("qa_id"):
                    by_qa[r["qa_id"]] = r
    return cases, by_qa


def write_case_files(out_root, case):
    cdir = os.path.join(out_root, "cases", case["safe_key"])
    os.makedirs(cdir, exist_ok=True)
    L.write_json_atomic(os.path.join(cdir, "case.json"), case)
    with open(os.path.join(cdir, "case.md"), "w") as f:
        f.write(L.case_markdown(case))
    return cdir


def write_report(out_root, cases, summary_lines, sel):
    with open(os.path.join(out_root, "report.html"), "w") as f:
        f.write(L.report_html(cases, out_root, summary_lines))
    with open(os.path.join(out_root, "summary.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["video_key", "safe_key", "qa_id", "source", "question_type", "question", "model_raw", "model_parsed",
                    "reference", "verdict", "score", "metric", "evidence", "category", "confidence", "review_stage",
                    "video_file"])
        for c in cases:
            q, s, r = c["qa"], c["score"], c.get("review") or {}
            w.writerow([q["video_key"], c["safe_key"], q["qa_id"], q["source_dataset"], q["question_type"], q["question"],
                        s["raw_answer"], s.get("parsed"), s["reference"], s["verdict"], s.get("score"), s.get("metric"),
                        r.get("evidence", ""), r.get("category", ""), r.get("confidence", ""),
                        r.get("stage", "not reviewed"), L.rel_link(c["video_file"], out_root)])


def missing_media(path):
    rows, _ = L.read_jsonl(path)
    return {r["video_key"] for r in rows}


def log_missing_media(path, video_key, archive, member):
    L.Journal(path).append({"video_key": video_key, "archive": archive, "member": member,
                            "logged": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})

`vsi590k_worker.py`

In [ ]:
%%writefile /content/vsi590k/vsi590k_worker.py
"""Cambrian-P inference worker for the VSI-590K failure review. Runs inside the Cambrian-P venv.

Loads the model once, then for each video in the batch file decodes the sampled frames once and answers that
video's selected questions. Every answer is appended (fsync) to a local journal at once; after each video its rows
are written to a durable shard with a completion marker. Model/preprocessing/prompt code is reused from the source
notebook's run_videos.py.
"""
import argparse
import json
import os
import sys
import time
import traceback

os.environ["MPLBACKEND"] = "Agg"   # Colab's inline backend leaks in through the environment; this script saves files only
import numpy as np
import torch
from PIL import Image

ap = argparse.ArgumentParser()
ap.add_argument("--batch", required=True, help="JSON: {videos: [{video_key, safe_key, local_path, qa: [...]}, ...]}")
ap.add_argument("--model_path", required=True)
ap.add_argument("--num_frames", type=int, required=True)
ap.add_argument("--config_hash", required=True)
ap.add_argument("--identity", required=True, help="JSON of the run identity, copied into every row")
ap.add_argument("--journal", required=True)
ap.add_argument("--errors", required=True)
ap.add_argument("--shard_dir", required=True)
ap.add_argument("--status", required=True)
ap.add_argument("--lib_dir", required=True)
ap.add_argument("--max_retries", type=int, default=2)
ap.add_argument("--stop_after", type=int, default=0, help="test only: exit after this many new answers")
args = ap.parse_args()
sys.path.insert(0, args.lib_dir)
import vsi590k_lib as L  # noqa: E402

identity = json.loads(args.identity)
GEN = dict(identity["GENERATION"])
journal, errors = L.Journal(args.journal), L.Journal(args.errors)
status = {"started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "answers": 0, "errors": 0, "videos": 0}


def save_status(**kw):
    status.update(kw)
    L.write_json_atomic(args.status, status)


# ---------------------------------------------------------------------------------------- model (from run_videos.py)
import cambrianp.utils as cambrianp_utils  # noqa: E402
from lmms_eval.models.simple.cambrian_p import CambrianP, expand2square  # noqa: E402
from cambrianp.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX  # noqa: E402
from cambrianp.conversation import conv_templates  # noqa: E402
from cambrianp.mm_utils import tokenizer_image_token  # noqa: E402


# Workaround kept from the source notebook, re-checked against the pinned code (b3c1552) and checkpoint:
# cambrianp/model/builder.py:230 still re-casts the config of any path containing "cambrian" to LLaVA-OneVision
# style, which overwrites vocab_size and deletes the camera-head keys; and the checkpoint's rec_model_path is
# JSON null, which VGGTReconstructor would try to torch.load. Keep the saved config, enable the camera head.
def _keep_checkpoint_config(config, model_path):
    config.load_rec_model = True
    if not os.path.isfile(str(getattr(config, "rec_model_path", None))):
        config.rec_model_path = "None"
    return config


cambrianp_utils.cast_cambrian_config_to_llava_ov_style = _keep_checkpoint_config
save_status(phase="loading model")
m = CambrianP(pretrained=args.model_path, model_name="llava_qwen", conv_template="qwen_1_5",
              max_frames_num=args.num_frames, use_camera_tokens=True, camera_tokens_mode="camera_tokens",
              camera_tokens_place="append_to_frame", query_mode="query_after_image")
pad_id = m.tokenizer.pad_token_id if m.tokenizer.pad_token_id is not None else m.tokenizer.eos_token_id

# Check the camera tokens and trained head really loaded (the failure mode the workaround exists for).
inner = m.model.get_model()
rec_head = getattr(inner, "rec_head", None)
cam = getattr(rec_head, "camera_tokens", None) if rec_head is not None else None
head_ok = rec_head is not None and cam is not None and float(cam.detach().float().abs().sum()) > 0
save_status(phase="model loaded", camera_head_loaded=bool(head_ok),
            camera_tokens_shape=list(cam.shape) if cam is not None else None,
            vocab_size=int(m.model.config.vocab_size), gpu=torch.cuda.get_device_name(0),
            torch=torch.__version__, cuda=torch.version.cuda)
if not head_ok:
    raise SystemExit("Camera tokens / trained head did not load: refusing to run (see the model log above).")


def ask(pixels, image_size, video, question):
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + question)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), m.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(m.device)
    return m._run_inference_step(ids, ids.ne(pad_id), [pixels], image_size, "video", dict(GEN), video)[0]


# ---------------------------------------------------------------------------------------- frames with provenance
def decode(path, k):
    """Decode exactly the frames the wrapper's load_video would sample, and say which they are."""
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(path, ctx=cpu(0), num_threads=0)
        total, fps = len(vr), float(vr.get_avg_fps())
        idx = L.sample_indices(total, k)
        frames = vr.get_batch(idx).asnumpy()
        reader = "decord"
    except Exception as exc:  # same fallback idea as the wrapper; recorded so it can be judged
        import av
        with av.open(path) as c:
            s = c.streams.video[0]
            all_frames = [f.to_ndarray(format="rgb24") for f in c.decode(s)]
            fps = float(s.average_rate or 0) or 30.0
        total = len(all_frames)
        idx = L.sample_indices(total, k)
        frames = np.stack([all_frames[i] for i in idx])
        reader = f"pyav (decord failed: {type(exc).__name__})"
    times = [i / fps for i in idx]
    return frames, {"frame_indices": idx, "timestamps_s": times, "total_frames": total, "fps": fps, "reader": reader,
                    "num_frames": k, "frame_size_hw": list(frames.shape[1:3])}


def preprocess(raw):
    bg = tuple(int(x * 255) for x in m._image_processor.image_mean)
    frames = [expand2square(Image.fromarray(f, mode="RGB"), bg) for f in raw]
    pixels = m._image_processor.preprocess(frames, return_tensors="pt")["pixel_values"].half().cuda()
    desc = (f"pad to square with mean color {bg} (expand2square), then the SigLIP image processor "
            f"(resize to {tuple(pixels.shape[-2:])}, normalize); fp16")
    return pixels, desc


# ---------------------------------------------------------------------------------------- main loop
batch = json.load(open(args.batch))
answered = 0
for vk, video in enumerate(batch["videos"], 1):
    rows = []
    save_status(phase=f"video {vk}/{len(batch['videos'])}", current=video["video_key"])
    t0 = time.time()
    try:
        raw, prov = decode(video["local_path"], args.num_frames)
        pixels, prov["preprocessing"] = preprocess(raw)
        image_size = list(raw.shape[1:3])
    except Exception:
        err = traceback.format_exc().strip().splitlines()[-1]
        for q in video["qa"]:
            e = {"kind": "answer", "qa_id": q["qa_id"], "video_key": video["video_key"], "status": "error",
                 "error": f"decode: {err}", "attempt": 1, "config_hash": args.config_hash,
                 "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            errors.append(e); rows.append(e)
        L.write_shard(args.shard_dir, video["safe_key"] + f"-{int(time.time())}", rows, args.config_hash)
        status["errors"] += len(video["qa"])
        continue
    meta = dict({"kind": "video_meta", "video_key": video["video_key"], "safe_key": video["safe_key"],
                 "config_hash": args.config_hash, "decode_seconds": round(time.time() - t0, 2)}, **prov)
    rows.append(meta)
    journal.append(meta)
    m.model.rec_head_preds = None          # poses are cached per generate(); not used here, cleared per video
    for q in video["qa"]:
        last_err = None
        for attempt in range(1, args.max_retries + 2):
            t1 = time.time()
            try:
                answer = ask(pixels, image_size, video["local_path"], q["prompt"])
                r = {"kind": "answer", "qa_id": q["qa_id"], "record_id": q["record_id"], "video_key": video["video_key"],
                     "status": "ok", "raw_answer": answer, "reference_answer": q["reference_answer"],
                     "attempt": attempt, "elapsed_s": round(time.time() - t1, 2), "config_hash": args.config_hash,
                     "identity": identity, "frame_provenance": "see video_meta row of this video",
                     "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                journal.append(r); rows.append(r)
                answered += 1
                status["answers"] += 1
                break
            except torch.cuda.OutOfMemoryError:
                e = {"kind": "answer", "qa_id": q["qa_id"], "video_key": video["video_key"], "status": "error",
                     "error": "CUDA out of memory", "attempt": attempt, "config_hash": args.config_hash,
                     "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                errors.append(e); rows.append(e)
                L.write_shard(args.shard_dir, video["safe_key"] + f"-{int(time.time())}", rows, args.config_hash)
                save_status(phase="stopped: out of GPU memory")
                raise SystemExit(f"Out of GPU memory at {args.num_frames} frames on {video['video_key']}. Frames and "
                                 "model are not changed automatically: use a GPU with more memory, or start a new "
                                 "RUN_ID with different settings.")
            except Exception:
                last_err = traceback.format_exc().strip().splitlines()[-1]
                e = {"kind": "answer", "qa_id": q["qa_id"], "video_key": video["video_key"], "status": "error",
                     "error": last_err, "attempt": attempt, "config_hash": args.config_hash,
                     "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                errors.append(e)
                if attempt == args.max_retries + 1:
                    rows.append(e)
                    status["errors"] += 1
        if args.stop_after and answered >= args.stop_after:
            save_status(phase=f"stopped on purpose after {answered} answers (resume test)")
            sys.exit(75)  # the video's shard is NOT written: its answers live only in the journal
    L.write_shard(args.shard_dir, video["safe_key"] + f"-{int(time.time())}", rows, args.config_hash)
    status["videos"] += 1
    torch.cuda.empty_cache()
save_status(phase="done", finished=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))

`vsi590k_frames.py`

In [ ]:
%%writefile /content/vsi590k/vsi590k_frames.py
"""Evidence frames for one failure case, decoded with the same reader as the model (decord, PyAV fallback).

input mode: the recorded model-input frames (exact indices from the run) at full resolution, plus contact sheets.
extra mode: diagnostic frames at --fps over the whole video and/or denser windows, kept apart from model inputs.
"""
import argparse
import json
import os
import sys

import numpy as np
from PIL import Image, ImageDraw

ap = argparse.ArgumentParser()
ap.add_argument("--video", required=True)
ap.add_argument("--out_dir", required=True)
ap.add_argument("--lib_dir", required=True)
ap.add_argument("--mode", choices=["input", "extra"], required=True)
ap.add_argument("--provenance", help="input mode: JSON file with frame_indices and timestamps_s from the run")
ap.add_argument("--fps", type=float, default=1.0, help="extra mode: frames per second over the whole video (0 = none)")
ap.add_argument("--window", action="append", default=[], help="extra mode: start,end,fps in seconds (repeatable)")
ap.add_argument("--max_extra", type=int, default=600)
args = ap.parse_args()
sys.path.insert(0, args.lib_dir)
import vsi590k_lib as L  # noqa: E402


def open_reader(path):
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(path, ctx=cpu(0), num_threads=0)
        return len(vr), float(vr.get_avg_fps()), lambda idx: vr.get_batch(idx).asnumpy(), "decord"
    except Exception as exc:
        import av
        with av.open(path) as c:
            s = c.streams.video[0]
            frames = [f.to_ndarray(format="rgb24") for f in c.decode(s)]
            fps = float(s.average_rate or 0) or 30.0
        return len(frames), fps, lambda idx: np.stack([frames[i] for i in idx]), f"pyav ({type(exc).__name__})"


total, fps, get, reader = open_reader(args.video)
os.makedirs(args.out_dir, exist_ok=True)

if args.mode == "input":
    prov = json.load(open(args.provenance))
    idx, times = prov["frame_indices"], prov["timestamps_s"]
    if total != prov["total_frames"]:
        raise SystemExit(f"Frame count differs from the run ({total} vs {prov['total_frames']}): these would not be the "
                         "frames the model saw.")
    thumbs = []
    for start in range(0, len(idx), 16):
        frames = get(idx[start:start + 16])
        for k, f in enumerate(frames):
            pos = start + k
            Image.fromarray(f).save(os.path.join(args.out_dir, L.frame_name(pos, idx[pos], times[pos])), quality=92)
            t = Image.fromarray(f)
            t.thumbnail((240, 240))
            thumbs.append((pos, idx[pos], times[pos], t))
    per_sheet, cols = 32, 8
    for s in range(0, len(thumbs), per_sheet):
        part = thumbs[s:s + per_sheet]
        w, h = max(t.width for *_, t in part), max(t.height for *_, t in part) + 16
        sheet = Image.new("RGB", (cols * w, -(-len(part) // cols) * h), "white")
        draw = ImageDraw.Draw(sheet)
        for k, (pos, fi, sec, t) in enumerate(part):
            x, y = (k % cols) * w, (k // cols) * h
            sheet.paste(t, (x, y + 16))
            draw.text((x + 2, y + 2), f"#{pos} f{fi} {sec:.2f}s", fill=(0, 0, 0))
        sheet.save(os.path.join(args.out_dir, f"sheet_{s // per_sheet + 1:02d}_inputs_{part[0][0]:03d}-{part[-1][0]:03d}.jpg"), quality=88)
    print(json.dumps({"written": len(thumbs), "reader": reader, "total_frames": total, "fps": fps}))
else:
    want = set()
    if args.fps > 0:
        step = max(1, int(round(fps / args.fps)))
        want.update(range(0, total, step))
    for w in args.window:
        a, b, wf = (float(x) for x in w.split(","))
        step = max(1, int(round(fps / wf)))
        want.update(range(max(0, int(a * fps)), min(total, int(b * fps) + 1), step))
    idx = sorted(want)[:args.max_extra]
    for start in range(0, len(idx), 32):
        for k, f in enumerate(get(idx[start:start + 32])):
            fi = idx[start + k]
            name = L.frame_name(start + k, fi, fi / fps).replace("in", "x", 1)
            Image.fromarray(f).save(os.path.join(args.out_dir, name), quality=88)
    print(json.dumps({"written": len(idx), "reader": reader, "total_frames": total, "fps": fps,
                      "capped": len(want) > args.max_extra}))

`vsi590k_selftest.py`

In [ ]:
%%writefile /content/vsi590k/vsi590k_selftest.py
"""Lightweight checks for vsi590k_lib (no GPU or network). Run: python vsi590k_selftest.py [path/to/vsi_590k.jsonl]"""
import io
import json
import os
import sys
import tarfile
import tempfile

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import vsi590k_lib as L

REV = "346fbd4e41dec974bf24894d0541a49327ee6669"
RESULTS = []


def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond), detail))
    print(("PASS " if cond else "FAIL ") + name + (f"  ({detail})" if detail and not cond else ""))


def rec(video, q, a, qtype="relative_direction_object", **extra):
    return dict({"conversations": [{"from": "human", "value": "<image>\n" + q}, {"from": "gpt", "value": a}],
                 "question_type": qtype, "video": video}, **extra)


MCQ = ("These are frames of a video.\nWhich is closest to the door?\nOptions:\nA. window\nB. table\nC. chair\n"
       "Answer with the option's letter from the given choices directly.")


def test_selection():
    a = L.select_indices(374148, 0.10, 42)
    b = L.select_indices(374148, 0.10, 42)
    check("1 selection is deterministic for the same seed", a == b)
    check("1 selection size is ceil(10% of the full population)", len(a) == 37415, str(len(a)))
    check("1 selection has no duplicates and stays in range", len(set(a)) == len(a) and 0 <= a[0] and a[-1] < 374148)
    check("1 a different seed gives a different selection", a != L.select_indices(374148, 0.10, 43))


def test_ids_and_grouping():
    r1 = rec("scannet/scene0001_00.mp4", MCQ, "A")
    r2 = rec("s3dis/area_1/scene0001_00.mp4", MCQ, "B")          # same basename, different source/folder
    rows1, _ = L.normalize_record(r1, 10, "vsi_590k.jsonl", REV)
    rows1b, _ = L.normalize_record(r1, 10, "vsi_590k.jsonl", REV)
    rows2, _ = L.normalize_record(r2, 11, "vsi_590k.jsonl", REV)
    check("2 qa ids are stable across runs", rows1[0]["qa_id"] == rows1b[0]["qa_id"])
    check("2 the same text on another line gets another id", rows1[0]["qa_id"] != L.normalize_record(r1, 12, "vsi_590k.jsonl", REV)[0][0]["qa_id"])
    check("2 duplicate basenames from different folders stay distinct",
          rows1[0]["video_key"] != rows2[0]["video_key"] and rows1[0]["safe_key"] != rows2[0]["safe_key"])
    g = L.group_by_video(rows1 + rows2 + L.normalize_record(rec("scannet/scene0001_00.mp4", MCQ, "C"), 13, "vsi_590k.jsonl", REV)[0])
    check("2 grouping by full video path keeps repeated videos together", len(g) == 2 and len(g["scannet/scene0001_00.mp4"]) == 2)


def test_prompt_excludes_reference():
    r = rec("scannet/scene0002_00.mp4", "These are frames of a video.\nHow many chairs?\nPlease answer the question "
            "using a single word or phrase.", "UNIQUE_REFERENCE_7731", "absolute_count")
    rows, exc = L.normalize_record(r, 1, "vsi_590k.jsonl", REV)
    check("3 the reference answer never appears in the prompt", "UNIQUE_REFERENCE_7731" not in rows[0]["prompt"])
    check("3 prompt is the human turn minus its single leading <image> token",
          L.prompt_excludes_reference(rows[0], r) and "<image>" not in rows[0]["prompt"])
    mt = {"conversations": [{"from": "human", "value": "<image>\nQ1"}, {"from": "gpt", "value": "A1"},
                            {"from": "human", "value": "Q2"}, {"from": "gpt", "value": "A2"}],
          "question_type": "relative_direction_object", "video": "scannet/x.mp4"}
    rows, exc = L.normalize_record(mt, 2, "vsi_590k.jsonl", REV)
    check("3 multi-turn video records are excluded with a reason, not truncated", not rows and exc["reason"] == "multi_turn_unsupported")
    img = {"conversations": r["conversations"], "question_type": "absolute_count", "image": "hypersim/a/b/c/d.jpg"}
    check("3 image records are outside the video population", L.normalize_record(img, 3, "vsi_590k.jsonl", REV)[1]["reason"] == "outside_population")


def make_tar_gz(members):
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w:gz") as tf:
        for name, data in members:
            if data is None:
                ti = tarfile.TarInfo(name); ti.type = tarfile.DIRTYPE; tf.addfile(ti)
            else:
                ti = tarfile.TarInfo(name); ti.size = len(data); tf.addfile(ti, io.BytesIO(data))
    buf.seek(0)
    return buf


def test_media():
    check("4 archive and member come from the annotation path",
          L.archive_for("s3dis/area_4/Area_4_office_1.mp4") == ("s3dis.tar.gz", "s3dis/area_4/Area_4_office_1.mp4"))
    with tempfile.TemporaryDirectory() as d:
        buf = make_tar_gz([("scannet", None), ("scannet/a.mp4", b"x" * 100), ("scannet/b.mp4", b"y" * 50),
                           ("scannet/c.mp4", b"z" * 70)])
        out = L.extract_members(buf, {"scannet/b.mp4"}, d, log=lambda *_: None)
        check("4 extracts only the wanted member, and stops once all are found",
              set(out["written"]) == {"scannet/b.mp4"} and out["stopped"] == "all_found" and
              open(os.path.join(d, "scannet/b.mp4"), "rb").read() == b"y" * 50 and not os.path.exists(os.path.join(d, "scannet/a.mp4")))
        buf = make_tar_gz([("scannet/a.mp4", b"x" * 100), ("scannet/b.mp4", b"y" * 50), ("scannet/c.mp4", b"z" * 70)])
        out = L.extract_members(buf, {"scannet/a.mp4", "scannet/c.mp4"}, d, byte_budget=120, log=lambda *_: None)
        check("4 respects the byte budget (stops before a member that would exceed it)",
              set(out["written"]) == {"scannet/a.mp4"} and out["stopped"] == "byte_budget")
        for bad in ("../evil.mp4", "/etc/passwd", "scannet/../../evil.mp4"):
            try:
                L.safe_extract_path(d, bad); ok = False
            except ValueError:
                ok = True
            check(f"4 refuses unsafe member path {bad!r}", ok)
        buf = make_tar_gz([("scannet/a.mp4", b"x")])
        out = L.extract_members(buf, {"scannet/missing.mp4"}, d, log=lambda *_: None)
        check("4 reports a member that is not in the archive", not out["written"] and out["stopped"] == "end_of_archive")


def test_resume():
    with tempfile.TemporaryDirectory() as d:
        sd = os.path.join(d, "predictions")
        cfg = "cfgA"
        L.write_shard(sd, "v1", [{"kind": "answer", "qa_id": "q1", "status": "ok", "config_hash": cfg, "raw_answer": "A", "answered_at": "2026-01-01T00:00:00Z"}], cfg)
        L.write_shard(sd, "v2", [{"kind": "answer", "qa_id": "q2", "status": "error", "config_hash": cfg}], cfg)
        open(os.path.join(sd, "v3.jsonl"), "w").write('{"kind":"answer","qa_id":"q3","status":"ok","config_hash":"cfgA"}\n')  # no marker
        L.write_shard(sd, "v4", [{"kind": "answer", "qa_id": "q4", "status": "ok", "config_hash": cfg}], cfg)
        with open(os.path.join(sd, "v4.jsonl"), "a") as f:
            f.write('{"kind":"answer","qa_id":"q9"')                                                      # corrupted after marker
        L.write_shard(sd, "v5", [{"kind": "answer", "qa_id": "q5", "status": "ok", "config_hash": "cfgB"}], "cfgB")
        rows, rep = L.load_shards(sd, cfg)
        ok, failed, _ = L.merge_results(rows, cfg)
        check("5 completed answers are kept", set(ok) == {"q1"})
        check("5 failed attempts are retried, not counted as done", set(failed) == {"q2"})
        check("5 incomplete, corrupted and other-config shards are ignored and reported",
              rep["incomplete"] == ["v3"] and rep["corrupted"] == ["v4"] and rep["other_config"] == ["v5"])
        L.write_shard(sd, "v2-retry", [{"kind": "answer", "qa_id": "q2", "status": "ok", "config_hash": cfg}], cfg)
        L.write_shard(sd, "v1-dup", [{"kind": "answer", "qa_id": "q1", "status": "ok", "config_hash": cfg, "raw_answer": "B", "answered_at": "2026-01-02T00:00:00Z"}], cfg)
        ok, failed, _ = L.merge_results(L.load_shards(sd, cfg)[0], cfg)
        check("5 a retried question is merged once, and duplicates do not double-count",
              set(ok) == {"q1", "q2"} and not failed and ok["q1"]["raw_answer"] == "A")
        j = L.Journal(os.path.join(d, "journal.jsonl"))
        j.append({"qa_id": "a"}); j.append({"qa_id": "b"})
        with open(j.path, "a") as f:
            f.write('{"qa_id": "c"')
        rows, bad = L.read_jsonl(j.path)
        check("5 a journal with a truncated last line keeps every complete row", [r["qa_id"] for r in rows] == ["a", "b"] and bad == 1)
        ident = {k: "x" for k in ("DATASET_REPO", "DATASET_REVISION", "MODEL_REPO", "MODEL_REVISION", "CODE_COMMIT",
                                   "SUBSET_FRACTION", "SEED", "NUM_FRAMES", "POPULATION", "PROMPT_POLICY", "GENERATION",
                                   "PREPROCESSING")}
        L.check_run_config(os.path.join(d, "run_config.json"), ident)
        try:
            L.check_run_config(os.path.join(d, "run_config.json"), dict(ident, NUM_FRAMES=64)); rejected = False
        except RuntimeError:
            rejected = True
        check("5 changed settings under the same RUN_ID are rejected", rejected)


def row(fmt, ref, qtype="relative_direction_object", prompt=MCQ, options=None):
    return {"qa_id": "q", "reference_answer": ref, "answer_format": fmt, "question_type": qtype, "prompt": prompt,
            "options": L.parse_options(prompt) if options is None else options}


def test_scoring():
    mc = row("mc_letter", "B")
    v = lambda r, a: L.score_answer(r, a)["verdict"]
    check("6 MC correct letter", v(mc, "B") == "correct")
    check("6 MC letter with text 'B. table'", v(mc, "B. table") == "correct")
    check("6 MC wrong letter", v(mc, "C") == "wrong")
    check("6 MC option text instead of letter", v(mc, "table") == "correct" and v(mc, "chair") == "wrong")
    check("6 MC reply that is no option is a format error, not wrong", v(mc, "I cannot tell") == "format_error")
    check("6 MC letter outside the options is a format error", v(mc, "E") == "format_error")
    check("6 MC reference given as 'C. 1' is read as letter C",
          v(row("mc_letter", "C. 1", "absolute_count", "Q?\nA. 3\nB. 4\nC. 1\nD. 2\nAnswer with the option's letter from the given choices directly."), "C") == "correct")
    num = row("numeric", "2.0", "absolute_distance_object", "These are frames of a video.\nHow far?", {})
    check("6 numeric within 5% is correct", v(num, "2.1") == "correct")
    check("6 numeric 50% off is imperfect (partial credit), not wrong", v(num, "2.9") == "imperfect")
    check("6 numeric more than 50% off is wrong", v(num, "3.2") == "wrong" and v(num, "0.9") == "wrong")
    check("6 numeric with units is parsed", v(num, "2 meters") == "correct")
    check("6 numeric reply without a number is a format error", v(num, "far") == "format_error")
    check("6 MRA boundary: exactly 5% off is full credit", L.mean_relative_accuracy(2.1, 2.0) == 1.0)
    check("6 MRA boundary: exactly 50% off keeps 0.1", L.mean_relative_accuracy(3.0, 2.0) == 0.1)
    ang = row("numeric", "350", "absolute_direction_object", "Angle?", {})
    check("6 angles wrap around 360", v(ang, "5") == "correct" and v(ang, "30") == "imperfect" and v(ang, "40") == "wrong" and v(ang, "200") == "wrong")
    d = row("closed_text", "left", "relative_direction_object", "Is the tv to the left or the right of the sofa?", {})
    check("6 closed-vocabulary direction", v(d, "left") == "correct" and v(d, "Right.") == "wrong")
    check("6 unclear direction goes to review", v(d, "it is near the sofa") == "needs_review")
    rd = row("closed_text", "dining table", "relative_distance_object",
             "which of these objects (dining table, toilet, window, chair) is the closest to the fridge?", {})
    check("6 listed-object answers", v(rd, "Dining table") == "correct" and v(rd, "toilet") == "wrong")
    check("6 semantically equivalent free text is not automatically wrong", v(rd, "the table") == "needs_review")
    ao = row("closed_text", "window, dining table, chair, fridge", "appearance_order",
             "categories in the video: chair, dining table, fridge, window?", {})
    check("6 appearance order", v(ao, "window, dining table, chair, fridge") == "correct"
          and v(ao, "chair, dining table, fridge, window") == "wrong" and v(ao, "window then table") == "needs_review")
    check("6 missing reference is unscorable", v(row("numeric", "", "absolute_count", "Q", {}), "3") == "unscorable")
    check("6 execution failure is not a wrong answer", L.score_answer(mc, "", status="error")["verdict"] == "execution_error")
    over = L.apply_review(L.score_answer(mc, "C"), {"final_verdict": "needs_review", "final_notes": "options ambiguous"})
    check("6 a reviewer override replaces the verdict and keeps the automatic basis", over["verdict"] == "needs_review" and over["reviewed"])


def test_frames_and_paths():
    idx = L.sample_indices(300, 128)
    check("7 frame indices match the model wrapper's np.linspace sampling",
          idx[0] == 0 and idx[-1] == 299 and len(idx) == 128 and idx == sorted(idx))
    check("7 short videos repeat frames rather than fail", L.sample_indices(10, 128)[:3] == [0, 0, 0])
    n = L.frame_name(5, 1234, 41.1333)
    p = L.parse_frame_name(n)
    check("7 frame names keep position, frame index and seconds as separate fields",
          p == {"set": "input", "position": 5, "frame_index": 1234, "seconds": 41.133})
    with tempfile.TemporaryDirectory() as d:
        os.makedirs(os.path.join(d, "c", "frames_input"))
        prov = {"frame_indices": [0, 1234], "timestamps_s": [0.0, 41.1333]}
        open(os.path.join(d, "c", "frames_input", n), "w").close()
        open(os.path.join(d, "c", "frames_input", L.frame_name(6, 999, 33.3)), "w").close()
        probs = L.check_case_images(os.path.join(d, "c"), prov)
        check("7 an image that is not one of the recorded input frames is flagged", len(probs) == 1 and "frame000999" in probs[0])
        check("7 report links are relative to the report folder",
              L.rel_link(os.path.join(d, "cases", "k", "case.md"), d) == "cases/k/case.md")
    page = L.report_html([], tempfile.gettempdir(), ["<script>alert(1)</script>"])
    check("7 report text is HTML-escaped", "<script>alert" not in page and "&lt;script&gt;" in page)


def test_failure_selection():
    qa = {}
    scores = []
    for k, (src, qt) in enumerate([("scannet", "absolute_count")] * 5 + [("adt", "appearance_order")] * 2 +
                                  [("procthor", "relative_direction_object")]):
        v = f"{src}/v{k // 2}.mp4"            # two wrong answers on some videos
        q = {"qa_id": f"q{k}", "video_key": v, "safe_key": L.safe_key(v), "source_dataset": src, "question_type": qt}
        qa[q["qa_id"]] = q
        scores.append({"qa_id": q["qa_id"], "verdict": "wrong"})
    scores.append({"qa_id": "q0", "verdict": "needs_review"})
    sel, res = L.select_failure_cases(scores, qa, target=20)
    keys = [c["video_key"] for c in sel]
    check("8 failures are counted by distinct video", len(keys) == len(set(keys)) == len({q["video_key"] for q in qa.values()}))
    check("8 selection is deterministic", sel == L.select_failure_cases(scores, qa, target=20)[0])
    check("8 the first picks span different sources", len({c["source_dataset"] for c in sel[:3]}) == 3)
    check("8 honest shortfall below the minimum",
          "NOT met" in L.shortfall_message(4, 20, 10) and "Shortfall" in L.shortfall_message(12, 20, 10))


def test_real_annotations(path):
    seen, rows, exc = 0, [], {}
    with open(path) as f:
        for line_no, line in enumerate(f, 1):
            r = json.loads(line)
            seen += 1
            got, e = L.normalize_record(r, line_no, "vsi_590k.jsonl", REV)
            rows += got
            if e:
                exc[e["reason"]] = exc.get(e["reason"], 0) + 1
            if got and not L.prompt_excludes_reference(got[0], r):
                check("real: prompt built from the human turn only", False, got[0]["qa_id"])
                return
    check("real: every record maps to a QA row or a documented exclusion",
          len(rows) + sum(exc.values()) == seen, f"{len(rows)} rows, exclusions {exc}")
    if L.sha256_file(path) == "025355ea328c0f1f73ba997fd52070a1f487248acd1fb9d8ac727886b319d231":
        check("real: video population size at the pinned revision", len(rows) == 374148, str(len(rows)))
    else:
        print("note: annotation file is not the pinned one; population-size check skipped")
    check("real: only image records are excluded", set(exc) == {"outside_population"}, str(exc))
    check("real: qa ids are unique", len({r["qa_id"] for r in rows}) == len(rows))
    fmts = {}
    for r in rows:
        fmts[r["answer_format"]] = fmts.get(r["answer_format"], 0) + 1
    unscorable = sum(1 for r in rows if L.score_answer(r, r["reference_answer"])["verdict"] != "correct")
    check("real: every reference scores as correct against itself (parser covers all formats)", unscorable == 0,
          f"{unscorable} rows don't; formats {fmts}")


if __name__ == "__main__":
    test_selection(); test_ids_and_grouping(); test_prompt_excludes_reference(); test_media(); test_resume()
    test_scoring(); test_frames_and_paths(); test_failure_selection()
    if len(sys.argv) > 1:
        test_real_annotations(sys.argv[1])
    failed = [n for n, ok, _ in RESULTS if not ok]
    print(f"\n{len(RESULTS) - len(failed)}/{len(RESULTS)} checks passed")
    sys.exit(1 if failed else 0)

In [ ]:
# @title 1c. Pin revisions and validate the run configuration
sys.path.insert(0, "/content/vsi590k")
import importlib, vsi590k_lib as L, vsi590k_pipeline as P
importlib.reload(L); importlib.reload(P)
assert PROMPT_POLICY == L.PROMPT_POLICY, "PROMPT_POLICY differs from the helper code"
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
pins_path = os.path.join(OUTPUT_ROOT, "pins.json")
pins = json.load(open(pins_path)) if os.path.exists(pins_path) else {}
access_problems = []

def resolve(name, given, fetch):
    """Explicit value > value saved for this run > fetched now. A value differing from the saved one is an error."""
    value = given or pins.get(name)
    if not value:
        try:
            value = fetch()
        except Exception as exc:
            access_problems.append(f"{name}: {type(exc).__name__}: {exc}")
            return ""
    if pins.get(name) and pins[name] != value:
        raise RuntimeError(f"{name} {value} differs from {pins[name]} saved for RUN_ID {RUN_ID}. "
                           "Use the saved value, or set a new RUN_ID for a new run.")
    pins[name] = value
    return value

def _dataset():
    info = api.dataset_info(DATASET_REPO)
    pins["DATASET_REPO_CANONICAL"] = info.id          # the repo answers under nyu-visionx/VSI-590K
    return info.sha
DATASET_REVISION = resolve("DATASET_REVISION", DATASET_REVISION, _dataset)
MODEL_REVISION = resolve("MODEL_REVISION", MODEL_REVISION, lambda: api.model_info(MODEL_REPO).sha)
CODE_COMMIT = resolve("CODE_COMMIT", CODE_COMMIT, lambda: subprocess.run(
    ["git", "ls-remote", REPO_URL, "HEAD"], capture_output=True, text=True, check=True).stdout.split()[0])
DATASET_REPO_CANONICAL = pins.get("DATASET_REPO_CANONICAL", DATASET_REPO)
if access_problems:
    print("Could not resolve:")
    for p in access_problems:
        print("  " + p)
    print(f"Check https://huggingface.co/datasets/{DATASET_REPO} and https://huggingface.co/{MODEL_REPO} in a browser "
          "(if a page asks you to accept terms, accept them and add HF_TOKEN in Colab secrets), then rerun this cell.")
    raise SystemExit("Revisions not pinned; nothing was changed.")
L.write_json_atomic(pins_path, pins)

IDENTITY = {"DATASET_REPO": DATASET_REPO_CANONICAL, "DATASET_REVISION": DATASET_REVISION, "MODEL_REPO": MODEL_REPO,
            "MODEL_REVISION": MODEL_REVISION, "CODE_COMMIT": CODE_COMMIT, "SUBSET_FRACTION": SUBSET_FRACTION,
            "SEED": SEED, "NUM_FRAMES": NUM_FRAMES, "POPULATION": POPULATION, "PROMPT_POLICY": PROMPT_POLICY,
            "GENERATION": GENERATION, "PREPROCESSING": PREPROCESSING}
run_config = L.check_run_config(os.path.join(OUTPUT_ROOT, "run_config.json"), IDENTITY)
CONFIG_HASH = run_config["config_hash"]
SHARD_DIR = os.path.join(OUTPUT_ROOT, "predictions")
JOURNAL = os.path.join(LOCAL_ROOT, "journal", f"answers-{CONFIG_HASH}.jsonl")
ERRORS_JOURNAL = os.path.join(LOCAL_ROOT, "journal", f"errors-{CONFIG_HASH}.jsonl")
for d in ("predictions", "reviews/cases", "reviews/qa", "logs", "failure_pool"):
    os.makedirs(os.path.join(OUTPUT_ROOT, d), exist_ok=True)
for k in ("DATASET_REVISION", "MODEL_REVISION", "CODE_COMMIT"):
    print(f"{k:17s} {IDENTITY[k]}")
print(f"config hash       {CONFIG_HASH}  (any change to data/model/code/frames/prompt/generation needs a new RUN_ID)")

## 2. Environment and model (reused from `cambrian_p_colab (3).ipynb`)
Same Python 3.11 venv and install steps as the source notebook's cell 3, with the repository checked out at `CODE_COMMIT`. No Cambrian-S. The model is downloaded at `MODEL_REVISION`.

**Drive cache.** The first session installs and downloads as usual, then saves the environment and the model weights under `CACHE_ROOT`. Later sessions restore them from Drive instead. A cached environment is used only if its name matches `CODE_COMMIT` and the install steps exactly, and only if it passes the same import and commit checks as a fresh install; otherwise it is ignored and the environment is reinstalled. Cached weights are used only for the same `MODEL_REVISION`, and every file's size is checked after copying.

In [ ]:
# @title 2a. Set up the environment and the model (restored from the Drive cache when possible)
import hashlib
REPO = "/content/cambrian-p"
VENV_PY = "/content/venv/bin/python"
MODEL_DIR = f"{REPO}/ckpts/Cambrian-P-7B"
CHECKOUT = r"""
set -e
cd /content
if [ ! -d cambrian-p/.git ]; then git clone -q "$REPO_URL" cambrian-p; fi
cd cambrian-p
if [ "$(git rev-parse HEAD)" != "$CODE_COMMIT" ]; then
    git fetch -q origin "$CODE_COMMIT" 2>/dev/null || git fetch -q origin
    git checkout -q "$CODE_COMMIT"
fi
echo "cambrian-p at $(git rev-parse HEAD)"
"""
INSTALL = r"""
set -e
cd /content/cambrian-p
echo "Installing (about 10-15 minutes) ..."
pip install -q uv
uv venv --seed --python 3.11 /content/venv
source /content/venv/bin/activate
pip install -q -e ./vggt
pip install -q hydra-core tensorboard iopath wcmatch fvcore
pip install -q --upgrade pip
pip install -q -e ".[train]"
pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --index-url https://download.pytorch.org/whl/cu121
pip install -q accelerate==0.29.3 easydict matplotlib roma evo imageio OpenEXR
pip install -q loguru sqlitedict sacrebleu evaluate tenacity pytz
pip install -q pyarrow==15.0.2
"""
# The cache name pins the code commit and the exact install steps: change either and the old copy is never used.
ENV_TAG = f"env-{CODE_COMMIT[:12]}-{hashlib.sha256(INSTALL.encode()).hexdigest()[:12]}"
ENV_TAR = os.path.join(CACHE_ROOT, "envs", ENV_TAG + ".tar")
MODEL_CACHE = os.path.join(CACHE_ROOT, "models", f"Cambrian-P-7B@{MODEL_REVISION}")
env = dict(os.environ, REPO_URL=REPO_URL, CODE_COMMIT=CODE_COMMIT, MODEL_REPO=MODEL_REPO, MODEL_REVISION=MODEL_REVISION,
           MODEL_DIR=MODEL_DIR, UV_PYTHON_INSTALL_DIR="/content/python", HF_HUB_ENABLE_HF_TRANSFER="1")
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN

def sh(script, what):
    if subprocess.run(["bash", "-c", script], env=env).returncode:
        raise SystemExit(f"{what} failed (see the log above).")

def env_ok():
    """The same checks whether the environment was just installed or restored from the cache."""
    if not os.path.exists(VENV_PY) or not os.path.isdir(os.path.join(REPO, ".git")):
        return False                       # nothing installed yet in this runtime
    imports = subprocess.run([VENV_PY, "-c", "import torch, decord"], capture_output=True, text=True)
    head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    return imports.returncode == 0 and head == CODE_COMMIT

def drive_free():
    try:
        return shutil.disk_usage("/content/drive/MyDrive").free
    except OSError:
        return 0

def tree_sizes(root):
    out = {}
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if x != ".cache"]
        for f in files:
            if not f.startswith(".revision-"):
                p = os.path.join(d, f)
                out[os.path.relpath(p, root)] = os.path.getsize(p)
    return out

# --- environment ---
t0 = L.time.time()
if env_ok():
    print("Environment already set up in this session.")
else:
    if CACHE_ENV and os.path.exists(ENV_TAR):
        print(f"Restoring the environment from the Drive cache ({os.path.getsize(ENV_TAR) / 1e9:.1f} GB, {ENV_TAG}) ...", flush=True)
        for d in ("venv", "python", "cambrian-p"):
            shutil.rmtree(os.path.join("/content", d), ignore_errors=True)
        r = subprocess.run(["tar", "-xf", ENV_TAR, "-C", "/content"])
        if r.returncode or not env_ok():
            print("The cached environment failed its checks; ignoring it and installing fresh.")
            for d in ("venv", "python", "cambrian-p"):
                shutil.rmtree(os.path.join("/content", d), ignore_errors=True)
    sh(CHECKOUT, "Checkout")
    if not env_ok():
        sh(INSTALL, "Install")
        if not env_ok():
            raise SystemExit("The environment failed its checks after installing (torch/decord import or commit).")
        if CACHE_ENV:
            print("Saving the environment to the Drive cache for later sessions ...", flush=True)
            os.makedirs(os.path.dirname(ENV_TAR), exist_ok=True)
            r = subprocess.run(["tar", "-cf", ENV_TAR + ".partial", "-C", "/content", "--exclude=cambrian-p/ckpts",
                                "venv", "python", "cambrian-p"])
            if r.returncode:
                print("  could not save it (Drive full?); the next session will reinstall.")
                if os.path.exists(ENV_TAR + ".partial"):
                    os.remove(ENV_TAR + ".partial")
            else:
                os.replace(ENV_TAR + ".partial", ENV_TAR)
                print(f"  saved {os.path.getsize(ENV_TAR) / 1e9:.1f} GB as {ENV_TAR}")
sh(CHECKOUT, "Checkout")
print(f"Environment ready in {(L.time.time() - t0) / 60:.1f} min.")

# --- model weights ---
t0 = L.time.time()
local_marker = os.path.join(MODEL_DIR, f".revision-{MODEL_REVISION}")
cache_marker = os.path.join(MODEL_CACHE, ".complete.json")
if os.path.exists(local_marker):
    print("Model already on local disk.")
else:
    if CACHE_MODEL and os.path.exists(cache_marker):
        print(f"Copying the model from the Drive cache ({MODEL_CACHE}) ...", flush=True)
        expected = json.load(open(cache_marker))["files"]
        shutil.rmtree(MODEL_DIR, ignore_errors=True)
        shutil.copytree(MODEL_CACHE, MODEL_DIR, ignore=shutil.ignore_patterns(".complete.json"))
        if tree_sizes(MODEL_DIR) != expected:
            print("  the cached copy is incomplete; downloading instead.")
            shutil.rmtree(MODEL_DIR, ignore_errors=True)
    if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
        sh("""cd /content/cambrian-p && /content/venv/bin/huggingface-cli download "$MODEL_REPO" --revision "$MODEL_REVISION" \
              --local-dir "$MODEL_DIR" """, "Model download")
        if CACHE_MODEL:
            files = tree_sizes(MODEL_DIR)
            need = sum(files.values())
            if drive_free() < need + 1e9:
                print(f"  not cached: Drive has {drive_free() / 1e9:.0f} GB free, the model needs {need / 1e9:.1f} GB.")
            else:
                print(f"Saving the model ({need / 1e9:.1f} GB) to the Drive cache for later sessions ...", flush=True)
                shutil.rmtree(MODEL_CACHE + ".partial", ignore_errors=True)
                shutil.copytree(MODEL_DIR, MODEL_CACHE + ".partial",
                                ignore=shutil.ignore_patterns(".cache", ".revision-*"))
                L.write_json_atomic(os.path.join(MODEL_CACHE + ".partial", ".complete.json"),
                                    {"model_repo": MODEL_REPO, "revision": MODEL_REVISION, "files": files})
                shutil.rmtree(MODEL_CACHE, ignore_errors=True)
                os.replace(MODEL_CACHE + ".partial", MODEL_CACHE)
    open(local_marker, "w").close()
print(f"Model ready in {(L.time.time() - t0) / 60:.1f} min.")

WORKER_ENV = dict(os.environ, PYTHONPATH=f"{REPO}/lmms-eval:{REPO}", PATH=f"/content/venv/bin:{os.environ['PATH']}",
                  MPLBACKEND="Agg")

def versions():
    freeze = subprocess.run([VENV_PY, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
    torch_info = subprocess.run([VENV_PY, "-c", "import torch;print(torch.__version__, torch.version.cuda)"],
                                capture_output=True, text=True).stdout.split()
    return {"pip_freeze": freeze, "torch": torch_info[:1], "cuda": torch_info[1:2], "gpu": gpu,
            "code_commit_checked_out": subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                                                      text=True).stdout.strip()}
ENV_INFO = versions()
assert ENV_INFO["code_commit_checked_out"] == CODE_COMMIT, "checked-out commit differs from CODE_COMMIT"

def update_manifest(**extra):
    path = os.path.join(OUTPUT_ROOT, "run_manifest.json")
    man = json.load(open(path)) if os.path.exists(path) else {"created": L.time.strftime("%Y-%m-%dT%H:%M:%SZ")}
    man.update({"run_id": RUN_ID, "identity": IDENTITY, "config_hash": CONFIG_HASH, "environment": ENV_INFO,
                "workarounds": ["cambrianp.utils.cast_cambrian_config_to_llava_ov_style replaced by a function that keeps "
                                "the checkpoint config, sets load_rec_model=True and rec_model_path='None' (checked "
                                "against builder.py:230 at the pinned commit and the checkpoint's config.json)"],
                "scoring_policy": L.SCORING_POLICY_VERSION, "parser": L.PARSER_VERSION, "lib": L.LIB_VERSION,
                "updated": L.time.strftime("%Y-%m-%dT%H:%M:%SZ")}, **extra)
    L.write_json_atomic(path, man)
update_manifest()
print("Environment ready; versions recorded in run_manifest.json. The model is loaded and checked in section 6.")

## 3. Discover the real VSI-590K annotations and media layout

In [ ]:
# @title 3a. List the repository at the pinned revision, fetch the card and annotations, check the schema
from huggingface_hub import hf_hub_download
# Everything here is cached on Drive per dataset revision: the file listing, the card and the verified annotation file.
DATA_CACHE = os.path.join(CACHE_ROOT, "datasets", f"VSI-590K@{DATASET_REVISION}")
os.makedirs(DATA_CACHE, exist_ok=True)
tree_path = os.path.join(DATA_CACHE, "files.json")
if os.path.exists(tree_path):
    FILES = json.load(open(tree_path))
else:
    FILES = {}
    for f in api.list_repo_tree(DATASET_REPO_CANONICAL, repo_type="dataset", revision=DATASET_REVISION, recursive=True):
        lfs = getattr(f, "lfs", None)
        FILES[f.path] = {"size": getattr(f, "size", None), "sha256": getattr(lfs, "sha256", None) if lfs else None}
    L.write_json_atomic(tree_path, FILES)
for p, info in sorted(FILES.items()):
    print(f"  {p:26s} {(info['size'] or 0) / 1e9:8.3f} GB")
ARCHIVE_SIZES = {p: i["size"] for p, i in FILES.items() if p.endswith(".tar.gz")}
ANNOTATION_FILES = sorted(p for p in FILES if p.endswith(".jsonl"))
print("Annotation files (all used to compute N):", ANNOTATION_FILES)

def cached_file(name, verify):
    """The Drive copy of a dataset file, downloading it once. A verified annotation file gets a .sha256 note so
    later sessions check its size and recorded hash instead of reading it again."""
    path = os.path.join(DATA_CACHE, name)
    note = path + ".sha256"
    if os.path.exists(path) and not verify:
        return path, None
    if os.path.exists(path) and os.path.exists(note) and os.path.getsize(path) == FILES[name]["size"]:
        recorded = open(note).read().strip()
        if recorded == (FILES[name]["sha256"] or recorded):
            return path, recorded
    tmp = hf_hub_download(DATASET_REPO_CANONICAL, name, repo_type="dataset", revision=DATASET_REVISION,
                          local_dir=os.path.join(LOCAL_ROOT, "meta"), token=HF_TOKEN)
    digest = L.sha256_file(tmp) if verify else None
    if verify and FILES[name]["sha256"] and digest != FILES[name]["sha256"]:
        raise SystemExit(f"{name}: SHA-256 {digest} does not match the repository's {FILES[name]['sha256']}")
    shutil.copyfile(tmp, path + ".part")
    os.replace(path + ".part", path)
    if verify:
        open(note, "w").write(digest)
    return path, digest

card = open(cached_file("README.md", False)[0]).read()
print("Dataset card: " + " | ".join(l.strip("- ") for l in card.splitlines() if l.startswith("- **")))
ANNOTATION_PATHS, ANNOTATION_SHA = [], {}
for name in ANNOTATION_FILES:
    path, digest = cached_file(name, True)
    if FILES[name]["sha256"] and digest != FILES[name]["sha256"]:
        raise SystemExit(f"{name}: recorded SHA-256 {digest} does not match the repository's {FILES[name]['sha256']}")
    ANNOTATION_PATHS.append(path)
    ANNOTATION_SHA[name] = digest
    print(f"{name}: sha256 {digest} (verified)")

# Parsing all 590,667 records takes a few minutes. Once this run's sample is frozen, a summary made from these exact
# annotation files (same hashes, same parser and helper versions) is enough to verify it, so parsing is skipped.
SUMMARY_PATH = os.path.join(DATA_CACHE, f"population-{POPULATION}.json")
POP_SUMMARY = P.load_population_summary(SUMMARY_PATH, ANNOTATION_SHA, POPULATION)
QA_ALL = EXCLUSIONS = RAW_BY_RECORD = None
if POP_SUMMARY and os.path.exists(os.path.join(OUTPUT_ROOT, "selection.json")):
    PROFILE = POP_SUMMARY["profile"]
    print(f"\nUsing the saved population summary (annotations unchanged): {POP_SUMMARY['population_size']:,} QA rows on "
          f"{POP_SUMMARY['videos']:,} videos; exclusions {POP_SUMMARY['exclusions']}. Schema checks passed when it was made.")
else:
    QA_ALL, EXCLUSIONS, RAW_BY_RECORD, PROFILE = P.profile_annotations(ANNOTATION_PATHS, DATASET_REVISION, POPULATION)
    print(f"\nRecords: {PROFILE['records']:,}; media {PROFILE['media']}; QA turns per record {PROFILE['turns']}")
    print("Field sets:", PROFILE["key_sets"])
    print("Sources:", PROFILE["sources"])
    print(f"Population '{POPULATION}': {len(QA_ALL):,} QA rows; exclusions by reason:",
          {r: sum(1 for e in EXCLUSIONS if e["reason"] == r) for r in sorted({e["reason"] for e in EXCLUSIONS})})
    print("Question types:", PROFILE["question_types"])
    for src in L.VIDEO_SOURCES:
        ex = next(q for q in QA_ALL if q["source_dataset"] == src)
        print(f"\n[{src}] {ex['question_type']} ({ex['answer_format']})\n{ex['prompt']}\n  -> reference: {ex['reference_answer']!r}")
    problems = P.schema_checks(QA_ALL, EXCLUSIONS, PROFILE, RAW_BY_RECORD)
    if problems:
        raise SystemExit("Schema checks failed; nothing selected:\n  " + "\n  ".join(problems))
    print("\nSchema checks passed: every record maps to a QA row or an exclusion; prompts come from the human turn only; "
          "every reference parses under the scoring rules.")
    POP_SUMMARY = P.population_summary(QA_ALL, EXCLUSIONS, PROFILE, ANNOTATION_SHA, POPULATION)
    L.write_json_atomic(SUMMARY_PATH, POP_SUMMARY)

## 4. Select and freeze the 10% sample

In [ ]:
# @title 4a. Sample once (seed 42), persist, and verify on later runs
SEL, QA, NEW = P.freeze_selection(OUTPUT_ROOT, QA_ALL, RAW_BY_RECORD, EXCLUSIONS, IDENTITY, ANNOTATION_SHA, POP_SUMMARY)
QA_BY_ID = {q["qa_id"]: q for q in QA}
print(("New selection saved." if NEW else "Existing selection verified (not resampled)."))
print(f"Population: {SEL['population_size']:,} video records (image records outside it: {SEL['records_outside_population']:,})")
print(f"Selected: {SEL['selected_records']:,} records = {SEL['selected_records'] / SEL['population_size']:.4%} | "
      f"{SEL['selected_qa']:,} questions | {SEL['selected_videos']:,} of "
      f"{POP_SUMMARY['videos']:,} videos ({SEL['selected_videos'] / POP_SUMMARY['videos']:.0%})")
from collections import Counter
print("By source:", dict(Counter(q["source_dataset"] for q in QA)))
print("By type:  ", dict(Counter(q["question_type"] for q in QA)))

## 5. Plan the media

In [ ]:
# @title 5a. Map every selected video to its archive member; show what must be transferred
MANIFEST = P.media_manifest(QA, ARCHIVE_SIZES)
MANIFEST_BY_VIDEO = {m["video_key"]: m for m in MANIFEST}
L.write_jsonl_atomic(os.path.join(OUTPUT_ROOT, "media_manifest.jsonl"), MANIFEST)
plan = P.transfer_plan(MANIFEST, ARCHIVE_SIZES)
free = shutil.disk_usage("/content").free
print(f"{'archive':22s} {'GB':>8s} {'videos':>7s}")
for p in plan:
    print(f"{p['archive']:22s} {(p['archive_bytes'] or 0) / 1e9:8.1f} {p['videos']:7d}")
need = sum(p["archive_bytes"] or 0 for p in plan)
print(f"Full run: {need / 1e9:.0f} GB of archives must be read (streamed or downloaded). Selected videos' own sizes are "
      f"unknown until extracted. Largest archive {max((p['archive_bytes'] or 0) for p in plan) / 1e9:.0f} GB; "
      f"working batch at most {MEDIA_BATCH_GB} GB of videos; local free space {free / 1e9:.0f} GB.")
print("Strategy per archive: read a complete copy from the Drive cache (CACHE_ARCHIVES) or local disk if there is one; "
      "else download it (to Drive with CACHE_ARCHIVES, else to local disk when there is room for archive + batch + 20 GB); "
      "else stream it (re-streamed once per batch). Archives are processed one at a time, smallest first.")
print("Drive cache:", "on: each archive is downloaded once and kept" if CACHE_ARCHIVES else
      "off (local copies are lost when the runtime ends; set CACHE_ARCHIVES = True if Drive has room)")

## 6. Run inference (resumable)

In [ ]:
# @title 6a. Runner (defines functions; run before 6b/6c)
import urllib.request, time as _time
from huggingface_hub import hf_hub_url
MEDIA_DIR = os.path.join(LOCAL_ROOT, "media")
ARCHIVE_DIR = os.path.join(LOCAL_ROOT, "archives")
WORK_DIR = os.path.join(LOCAL_ROOT, "batches")
MISSING_LOG = os.path.join(OUTPUT_ROOT, "logs", "media_missing.jsonl")
for d in (MEDIA_DIR, ARCHIVE_DIR, WORK_DIR, os.path.dirname(JOURNAL)):
    os.makedirs(d, exist_ok=True)

ARCHIVE_CACHE = os.path.join(DATA_CACHE, "archives")

def open_archive(archive, prefer_download):
    """A readable stream of the archive: a complete copy on Drive (CACHE_ARCHIVES) or local disk if present;
    else a new copy if there is room for one; else HTTP streaming."""
    size = ARCHIVE_SIZES.get(archive) or 0
    for folder in ([ARCHIVE_CACHE] if CACHE_ARCHIVES else []) + [ARCHIVE_DIR]:
        path = os.path.join(folder, archive)
        if os.path.exists(path) and os.path.getsize(path) == size:
            print(f"  reading {archive} from {'the Drive cache' if folder == ARCHIVE_CACHE else 'local disk'}", flush=True)
            return open(path, "rb")
    if CACHE_ARCHIVES and drive_free() > size + 5e9:
        folder, where = ARCHIVE_CACHE, "the Drive cache (kept for later sessions)"
    elif prefer_download and shutil.disk_usage("/content").free - size - (MEDIA_BATCH_GB + 20) * 1e9 > 0:
        folder, where = ARCHIVE_DIR, "local disk"
    else:
        folder = None
    if folder:
        print(f"  downloading {archive} ({size / 1e9:.1f} GB) to {where} ...", flush=True)
        os.makedirs(folder, exist_ok=True)
        tmp = hf_hub_download(DATASET_REPO_CANONICAL, archive, repo_type="dataset", revision=DATASET_REVISION,
                              local_dir=os.path.join(folder, ".download"), token=HF_TOKEN)
        os.replace(tmp, os.path.join(folder, archive))
        return open(os.path.join(folder, archive), "rb")
    url = hf_hub_url(DATASET_REPO_CANONICAL, archive, repo_type="dataset", revision=DATASET_REVISION)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {})
    print(f"  streaming {archive} ...", flush=True)
    return urllib.request.urlopen(req, timeout=300)

def run_worker(batch_path, extra, journal=None, shard_dir=None, status_name="worker_status.json"):
    cmd = [VENV_PY, "/content/vsi590k/vsi590k_worker.py", "--batch", batch_path, "--model_path", "ckpts/Cambrian-P-7B",
           "--num_frames", str(NUM_FRAMES), "--config_hash", CONFIG_HASH, "--identity", json.dumps(IDENTITY),
           "--journal", journal or JOURNAL, "--errors", ERRORS_JOURNAL, "--shard_dir", shard_dir or SHARD_DIR,
           "--status", os.path.join(OUTPUT_ROOT, "logs", status_name), "--lib_dir", "/content/vsi590k",
           "--max_retries", str(MAX_RETRIES)] + extra
    log_path = os.path.join(OUTPUT_ROOT, "logs", f"worker-{_time.strftime('%Y%m%d-%H%M%S')}-{_time.time_ns() % 10**6:06d}.log")
    with open(log_path, "w") as logf:
        p = subprocess.Popen(cmd, cwd=REPO, env=WORKER_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            logf.write(line)
            if any(k in line for k in ("token_num:", "[cam-load]", "SUCCESS", "WARNING", "Error", "error", "Traceback")):
                print("   " + line.rstrip()[:220])
        code_ = p.wait()
    st = json.load(open(os.path.join(OUTPUT_ROOT, "logs", status_name))) if os.path.exists(os.path.join(OUTPUT_ROOT, "logs", status_name)) else {}
    print(f"  worker exit {code_}: {st.get('phase')} | answers {st.get('answers')} | camera head loaded: "
          f"{st.get('camera_head_loaded')} | log {log_path}")
    return code_

def keep_media(video_key, path):
    """Copy a video with a verified wrong answer to the Drive pool (capped), before its local copy is deleted."""
    pool = os.path.join(OUTPUT_ROOT, "failure_pool")
    if len(os.listdir(pool)) >= FAILURE_POOL_VIDEOS:
        return
    import glob
    answers = {}
    for shard in glob.glob(os.path.join(SHARD_DIR, MANIFEST_BY_VIDEO[video_key]["safe_key"] + "-*.jsonl")):
        for r in L.read_jsonl(shard)[0]:
            if r.get("status") == "ok" and r.get("config_hash") == CONFIG_HASH:
                answers[r["qa_id"]] = r["raw_answer"]
    wrong = [QA_BY_ID[k] for k, a in answers.items() if k in QA_BY_ID
             and L.score_answer(QA_BY_ID[k], a)["verdict"] == "wrong"]
    dest = os.path.join(pool, MANIFEST_BY_VIDEO[video_key]["safe_key"] + ".mp4")
    if wrong and not os.path.exists(dest):
        shutil.copyfile(path, dest + ".part")
        os.replace(dest + ".part", dest)

def context(prefer_download):
    return {"open_archive": open_archive, "run_worker": run_worker, "media_dir": MEDIA_DIR, "work_dir": WORK_DIR,
            "shard_dir": SHARD_DIR, "journal": JOURNAL, "config_hash": CONFIG_HASH,
            "batch_bytes": int(MEDIA_BATCH_GB * 1e9), "archive_sizes": ARCHIVE_SIZES, "keep_media": keep_media,
            "log_missing": lambda v, a, m: P.log_missing_media(MISSING_LOG, v, a, m),
            "skip_videos": P.missing_media(MISSING_LOG), "prefer_download": prefer_download}

PILOT_QA = L.pilot_subset(QA, ARCHIVE_SIZES, PILOT_VIDEOS, PILOT_MAX_QA)
print(f"Pilot: {len(PILOT_QA)} questions on {len({q['video_key'] for q in PILOT_QA})} videos "
      f"({', '.join(sorted({q['video_key'] for q in PILOT_QA}))}), from the smallest needed archive.")

In [ ]:
# @title 6b. Run (pilot or full, per RUN_MODE). Rerun to resume after any interruption.
subset = PILOT_QA if RUN_MODE == "pilot" else QA
t0 = _time.time()
summary = P.run_selection(subset, MANIFEST_BY_VIDEO, context(prefer_download=(RUN_MODE == "full")))
ok, failed, meta, report = P.gather_results(SHARD_DIR, JOURNAL, CONFIG_HASH)
done = sum(1 for q in subset if q["qa_id"] in ok)
print(f"\n{RUN_MODE}: {done}/{len(subset)} questions answered in this set ({len(ok)} for the whole run); "
      f"{(_time.time() - t0) / 60:.1f} min this session. Shards: {report['complete']} complete; ignored: "
      f"{len(report['incomplete'])} incomplete, {len(report['corrupted'])} corrupted, {len(report['other_config'])} other-config.")
if RUN_MODE == "pilot" and done:
    print("\nSample pilot answers:")
    for q in subset[:5]:
        if q["qa_id"] in ok:
            print(f"  [{q['question_type']}] model {ok[q['qa_id']]['raw_answer']!r} | reference {q['reference_answer']!r}")
update_manifest(last_run={"mode": RUN_MODE, "summary": summary, "answered_total": len(ok)})

In [ ]:
# @title 6c. Pilot resume check: interrupt on purpose, restart, and confirm no answer is lost or duplicated
# Uses a separate journal and shard folder, so the run's own results are untouched.
assert RUN_MODE == "pilot", "Run this check in pilot mode."
TEST_DIR = os.path.join(LOCAL_ROOT, "resume_check")
shutil.rmtree(TEST_DIR, ignore_errors=True)
os.makedirs(TEST_DIR)
test_journal, test_shards = os.path.join(TEST_DIR, "journal.jsonl"), os.path.join(TEST_DIR, "shards")
def run_test_worker(batch_path, extra, stop=0):
    return run_worker(batch_path, extra + (["--stop_after", str(stop)] if stop else []), journal=test_journal,
                      shard_dir=test_shards, status_name="resume_check_status.json")
ctx = dict(context(False), journal=test_journal, shard_dir=test_shards, keep_media=lambda v, p: None,
           run_worker=lambda b, e: run_test_worker(b, e, stop=1))
P.run_selection(PILOT_QA, MANIFEST_BY_VIDEO, ctx)
ok1, _, _, _ = P.gather_results(test_shards, test_journal, CONFIG_HASH)
first = {k: v["raw_answer"] for k, v in ok1.items()}
print(f"After the deliberate stop: {len(first)} answers kept (in the journal; that video's shard was not finished).")
ctx["run_worker"] = lambda b, e: run_test_worker(b, e)
P.run_selection(PILOT_QA, MANIFEST_BY_VIDEO, ctx)
ok2, _, _, _ = P.gather_results(test_shards, test_journal, CONFIG_HASH)
rows, _ = L.read_jsonl(test_journal)
per_q = Counter(r["qa_id"] for r in rows if r.get("kind") == "answer" and r.get("status") == "ok")
checks = {"the deliberate stop happened before all pilot questions were answered": 1 <= len(first) < len(PILOT_QA),
          "answers written before the stop were kept": len(first) >= 1,
          "all pilot questions answered after restart": all(q["qa_id"] in ok2 for q in PILOT_QA),
          "kept answers unchanged": all(ok2[k]["raw_answer"] == v for k, v in first.items()),
          "no question answered twice": bool(per_q) and max(per_q.values()) == 1}
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
update_manifest(resume_check=checks)

## 7. Score the answers

In [ ]:
# @title 7a. Merge results, score, and write predictions / errors / scores / coverage
ok, failed, META, report = P.gather_results(SHARD_DIR, JOURNAL, CONFIG_HASH)
REVIEWS_CASES, REVIEWS_QA = P.load_reviews(os.path.join(OUTPUT_ROOT, "reviews"))
MISSING = P.missing_media(MISSING_LOG)
SCORES = P.score_all(QA, ok, failed, REVIEWS_QA, MISSING)
P.write_score_outputs(OUTPUT_ROOT, ok, failed, SCORES, QA_BY_ID)
COVERAGE = P.coverage(SEL, QA, ok, failed, SCORES, MISSING, RUN_MODE)
COVERAGE["scoring"] = {"policy": L.SCORING_POLICY_VERSION, "parser": L.PARSER_VERSION,
                       "multiple choice": "option letter (or exact option text) vs reference letter",
                       "numbers": "VSI-Bench-style mean relative accuracy, used as a diagnostic (not confirmed as "
                                  "VSI-590K's metric): correct within 5%, wrong beyond 50%, else imperfect",
                       "angles": f"circular error: correct <= {L.ANGLE_TOLERANCE['correct_max_deg']:g} deg, "
                                 f"wrong > {L.ANGLE_TOLERANCE['wrong_min_deg']:g} deg (diagnostic thresholds)",
                       "ProcTHOR word answers": "closed vocabulary / listed objects / ordering; anything else needs review"}
L.write_json_atomic(os.path.join(OUTPUT_ROOT, "coverage.json"), COVERAGE)
print(json.dumps({k: COVERAGE[k] for k in ("selected", "completed", "failed_qa", "pending_qa", "verdicts", "full_run_complete")}, indent=1))
print("Wrong answers by source:", COVERAGE["wrong_by_source"])
print("Files: predictions.jsonl, errors.jsonl, scores.jsonl, coverage.json, failure_candidates.csv, "
      "imperfect_cases.csv, needs_review.csv")

## 8. Choose and export 10–20 distinct failure videos

In [ ]:
# @title 8a. Choose cases (one primary wrong question per video, diverse by source and type)
SELECTED, RESERVE = L.select_failure_cases(SCORES, QA_BY_ID, TARGET_FAILURE_VIDEOS)
L.write_json_atomic(os.path.join(OUTPUT_ROOT, "selected_failure_videos.json"),
                    {"selected": SELECTED, "reserve": RESERVE, "rule": "deterministic: least-represented source, then "
                     "question type, then SHA-1 of the video path", "only_verdict": "wrong"})
print(L.shortfall_message(len(SELECTED), TARGET_FAILURE_VIDEOS, MIN_FAILURE_VIDEOS))
if not COVERAGE["full_run_complete"]:
    print(f"Note: the full 10% run is not complete ({COVERAGE['completed']['qa']:,}/{SEL['selected_qa']:,} questions); "
          "cases come from the part processed so far.")
for c in SELECTED:
    print(f"  {c['source_dataset']:12s} {c['question_type']:28s} {c['video_key']}")

In [ ]:
# @title 8b. Export each case: playable video, exact input frames, contact sheets
def fetch_video(entry):
    """Path to the case video: Drive pool, local media, or re-fetched from its archive via the media manifest."""
    m = MANIFEST_BY_VIDEO[entry["video_key"]]
    for cand in (os.path.join(OUTPUT_ROOT, "failure_pool", m["safe_key"] + ".mp4"), os.path.join(MEDIA_DIR, m["member"])):
        if os.path.exists(cand):
            return cand
    print(f"  re-fetching {m['member']} from {m['archive']} ...", flush=True)
    f = open_archive(m["archive"], prefer_download=False)
    try:
        out = L.extract_members(f, {m["member"]}, MEDIA_DIR)
    finally:
        f.close()
    if m["member"] not in out["written"]:
        raise FileNotFoundError(f"{m['member']} not found in {m['archive']}")
    return out["written"][m["member"]]

CASES = []
for entry in SELECTED:
    q = QA_BY_ID[entry["qa_id"]]
    meta = META.get(entry["video_key"])
    cdir = os.path.join(OUTPUT_ROOT, "cases", entry["safe_key"])
    os.makedirs(os.path.join(cdir, "frames_input"), exist_ok=True)
    os.makedirs(os.path.join(cdir, "frames_extra"), exist_ok=True)
    vfile = os.path.join(OUTPUT_ROOT, "failure_videos", entry["safe_key"] + ".mp4")
    if not os.path.exists(vfile):
        src = fetch_video(entry)
        os.makedirs(os.path.dirname(vfile), exist_ok=True)
        shutil.copyfile(src, vfile + ".part")
        os.replace(vfile + ".part", vfile)
    prov_path = os.path.join(cdir, "provenance.json")
    L.write_json_atomic(prov_path, meta)
    if not any(f.startswith("sheet_") for f in os.listdir(os.path.join(cdir, "frames_input"))):
        r = subprocess.run([VENV_PY, "/content/vsi590k/vsi590k_frames.py", "--video", vfile, "--out_dir",
                            os.path.join(cdir, "frames_input"), "--lib_dir", "/content/vsi590k", "--mode", "input",
                            "--provenance", prov_path], env=WORKER_ENV, capture_output=True, text=True)
        if r.returncode:
            print(f"  frames failed for {entry['video_key']}: {r.stdout[-300:]}{r.stderr[-300:]}")
    score = next(s for s in SCORES if s["qa_id"] == entry["qa_id"])
    case = {"safe_key": entry["safe_key"], "video_key": entry["video_key"], "qa": q, "score": score,
            "provenance": meta, "video_file": vfile, "video_bytes": os.path.getsize(vfile),
            "video_sha256": L.sha256_file(vfile), "other_wrong_qa_ids": entry["other_wrong_qa_ids"],
            "review": REVIEWS_CASES.get(entry["safe_key"])}
    P.write_case_files(OUTPUT_ROOT, case)
    problems = L.check_case_images(cdir, meta)
    CASES.append(case)
    print(f"  {entry['safe_key']}: video {case['video_bytes'] / 1e6:.1f} MB, "
          f"{sum(1 for f in os.listdir(os.path.join(cdir, 'frames_input')) if not f.startswith('sheet_'))} input frames"
          + (f"; image problems: {problems}" if problems else ""))
L.write_json_atomic(os.path.join(OUTPUT_ROOT, "key_mapping.json"), {c["safe_key"]: c["video_key"] for c in CASES})
print(f"Exported {len(CASES)} case(s); mapping of safe names to dataset paths in key_mapping.json.")

In [ ]:
# @title 8c. Review cases (manual stage: blind judgment first, then reveal answers)
import ipywidgets as W
from IPython.display import display, clear_output
REVIEW_DIR = os.path.join(OUTPUT_ROOT, "reviews", "cases")

def load_review(c):
    p = os.path.join(REVIEW_DIR, c["safe_key"] + ".json")
    return json.load(open(p)) if os.path.exists(p) else {"safe_key": c["safe_key"], "qa_id": c["qa"]["qa_id"]}

def save_review(r):
    r["saved"] = L.time.strftime("%Y-%m-%dT%H:%M:%SZ")
    L.write_json_atomic(os.path.join(REVIEW_DIR, r["safe_key"] + ".json"), r)

UI = {}   # the current case's widgets (also lets a test drive the interface)
pick = W.Dropdown(options=[(f"{i + 1}. {c['video_key']}", i) for i, c in enumerate(CASES)], layout=W.Layout(width="70%"))
progress, body = W.HTML(), W.Output()

def refresh_progress():
    rs = [load_review(c) for c in CASES]
    progress.value = (f"<b>{sum(r.get('stage') == 'final' for r in rs)}/{len(CASES)}</b> final reviews, "
                      f"{sum(r.get('stage') == 'provisional' for r in rs)} provisional only")

def show(i):
    c, r = CASES[i], load_review(CASES[i])
    q = c["qa"]
    cdir = os.path.join(OUTPUT_ROOT, "cases", c["safe_key"])
    sheets = sorted(f for f in os.listdir(os.path.join(cdir, "frames_input")) if f.startswith("sheet_"))
    head = W.HTML(f"<h3>{L.esc(q['video_key'])}</h3><p>{L.esc(q['source_dataset'])} · {L.esc(q['question_type'])}</p>"
                  f"<p><b>Question:</b> {L.esc(q['question'])}</p><ul>" +
                  "".join(f"<li>{L.esc(k)}. {L.esc(v)}</li>" for k, v in sorted(q["options"].items())) + "</ul>"
                  f"<p>Video: <code>{L.esc(c['video_file'])}</code> (open it from Drive to watch)</p>")
    imgs = [W.Image(value=open(os.path.join(cdir, "frames_input", s), "rb").read(), format="jpg", layout=W.Layout(width="100%"))
            for s in sheets]
    prov_ans = W.Text(value=r.get("provisional_answer", ""), description="Your answer")
    prov_ev = W.Dropdown(options=L.EVIDENCE_JUDGMENTS, value=r.get("provisional_evidence", L.EVIDENCE_JUDGMENTS[0]),
                         description="Evidence")
    save_prov, reveal_box = W.Button(description="Save blind judgment"), W.Output()

    def do_reveal():
        with reveal_box:
            clear_output()
            s = c["score"]
            display(W.HTML(f"<p><b>Model:</b> {L.esc(s['raw_answer'])} → {L.esc(s.get('parsed'))} &nbsp; "
                           f"<b>Reference:</b> {L.esc(s['reference'])} &nbsp; <b>Auto verdict:</b> {L.esc(s['verdict'])}"
                           f"</p><p>{L.esc(s['basis'])}</p>"))
            ev = W.Dropdown(options=L.EVIDENCE_JUDGMENTS, value=r.get("evidence", r.get("provisional_evidence", L.EVIDENCE_JUDGMENTS[0])), description="Evidence")
            cat = W.Dropdown(options=L.FAILURE_CATEGORIES, value=r.get("category", "other"), description="Category")
            conf = W.Dropdown(options=["low", "medium", "high"], value=r.get("confidence", "low"), description="Confidence")
            verdict = W.Dropdown(options=[("confirm wrong", "wrong"), ("model was right (override)", "correct"),
                                          ("unclear: needs review", "needs_review")], value=r.get("final_verdict", "wrong"),
                                 description="Verdict")
            obs = W.Textarea(value=r.get("observations", ""), description="Seen", placeholder="what is visible, with times")
            hyp = W.Textarea(value=r.get("hypotheses", ""), description="Guesses", placeholder="tentative causes (attention, pose, ...)")
            notes = W.Textarea(value=r.get("final_notes", ""), description="Notes")
            win = W.Text(value="", description="Window", placeholder="start,end seconds for dense frames")
            extra_btn, save_final, msg = W.Button(description="Extract extra frames"), W.Button(description="Save final review", button_style="success"), W.HTML()

            def extra(_):
                args = ["--fps", "1"] + (["--window", win.value.strip() + ",5"] if win.value.strip() else [])
                rr = subprocess.run([VENV_PY, "/content/vsi590k/vsi590k_frames.py", "--video", c["video_file"], "--out_dir",
                                     os.path.join(cdir, "frames_extra"), "--lib_dir", "/content/vsi590k", "--mode", "extra"] + args,
                                    env=WORKER_ENV, capture_output=True, text=True)
                msg.value = L.esc((rr.stdout or rr.stderr)[-300:]) + " (saved to frames_extra/, separate from model inputs)"

            def final(_):
                r.update(stage="final", evidence=ev.value, category=cat.value, confidence=conf.value, final_verdict=verdict.value,
                         observations=obs.value, hypotheses=hyp.value, final_notes=notes.value)
                save_review(r)
                refresh_progress()
                msg.value = "Final review saved."
            extra_btn.on_click(extra)
            save_final.on_click(final)
            UI.update(evidence=ev, category=cat, confidence=conf, verdict=verdict, observations=obs, hypotheses=hyp,
                      notes=notes, window=win, extra=extra_btn, save_final=save_final, final_msg=msg)
            display(W.VBox([ev, cat, conf, verdict, obs, hyp, notes, W.HBox([win, extra_btn]), save_final, msg]))

    def provisional(_):
        r.update(stage=r.get("stage") if r.get("stage") == "final" else "provisional",
                 provisional_answer=prov_ans.value, provisional_evidence=prov_ev.value)
        save_review(r)
        refresh_progress()
        do_reveal()
    save_prov.on_click(provisional)
    UI.clear()
    UI.update(case=c, provisional_answer=prov_ans, provisional_evidence=prov_ev, save_provisional=save_prov,
              revealed=lambda: "evidence" in UI)
    with body:
        clear_output()
        display(W.VBox([head] + imgs + [W.HTML("<b>Blind judgment (answers hidden until saved)</b>"), prov_ans, prov_ev,
                                        save_prov, reveal_box]))
    if r.get("stage") in ("provisional", "final"):
        do_reveal()

pick.observe(lambda ch: show(ch["new"]) if ch["name"] == "value" else None, names="value")
refresh_progress()
display(W.VBox([progress, pick, body]))
if CASES:
    show(0)
else:
    print("No exported cases yet: run 7a, 8a and 8b first.")

In [ ]:
# @title 8d. (Optional) Review answers the scorer could not decide ("needs_review")
import ipywidgets as W
from IPython.display import display
NR = [s for s in SCORES if s["verdict"] == "needs_review" and not s.get("reviewed")]
QA_REVIEW_DIR = os.path.join(OUTPUT_ROOT, "reviews", "qa")
rows_ = []
for s in NR[:50]:
    q = QA_BY_ID[s["qa_id"]]
    v = W.Dropdown(options=[("leave undecided", ""), ("same meaning: correct", "correct"), ("different: wrong", "wrong")])
    def save(change, s=s, q=q):
        if change["new"]:
            L.write_json_atomic(os.path.join(QA_REVIEW_DIR, s["qa_id"] + ".json"),
                                {"qa_id": s["qa_id"], "final_verdict": change["new"], "final_notes": "manual meaning check",
                                 "saved": L.time.strftime("%Y-%m-%dT%H:%M:%SZ")})
    v.observe(save, names="value")
    rows_.append(W.VBox([W.HTML(f"<p><b>{L.esc(q['question'])}</b><br>model: {L.esc(s['raw_answer'])} | reference: "
                                f"{L.esc(s['reference'])}</p>"), v]))
print(f"{len(NR)} answers need a meaning check (showing up to 50). Rerun 7a afterwards to apply the decisions.")
display(W.VBox(rows_))

## 9. Write the final outputs

In [ ]:
# @title 9a. Report, summary and manifest
REVIEWS_CASES, REVIEWS_QA = P.load_reviews(os.path.join(OUTPUT_ROOT, "reviews"))
for c in CASES:
    c["review"] = REVIEWS_CASES.get(c["safe_key"])
    P.write_case_files(OUTPUT_ROOT, c)
final_n = sum(1 for c in CASES if (c.get("review") or {}).get("stage") == "final")
confirmed = [c for c in CASES if (c.get("review") or {}).get("final_verdict", "wrong") == "wrong"]
lines = [
    f"Run {RUN_ID}: {'pilot only' if RUN_MODE == 'pilot' and not COVERAGE['full_run_complete'] else ('full 10% run complete' if COVERAGE['full_run_complete'] else 'full 10% run in progress')}",
    f"Dataset {DATASET_REPO_CANONICAL}@{DATASET_REVISION[:12]}; model {MODEL_REPO}@{MODEL_REVISION[:12]}; code {CODE_COMMIT[:12]}; {NUM_FRAMES} frames",
    f"Sample: {SEL['fraction']:.0%} of {SEL['population_size']:,} video records (seed {SEL['seed']}): "
    f"{SEL['selected_records']:,} records, {SEL['selected_qa']:,} questions, {SEL['selected_videos']:,} videos",
    f"Completed: {COVERAGE['completed']['records']:,} records / {COVERAGE['completed']['qa']:,} questions / "
    f"{COVERAGE['completed']['videos_with_any_answer']:,} videos; execution errors {COVERAGE['failed_qa']:,}; pending {COVERAGE['pending_qa']:,}",
    f"Verdicts: {COVERAGE['verdicts']}",
    f"Verified wrong-answer videos exported: {len(CASES)} ({L.shortfall_message(len(CASES), TARGET_FAILURE_VIDEOS, MIN_FAILURE_VIDEOS)})",
    f"Review status: {final_n}/{len(CASES)} final; {len(CASES) - len(confirmed)} overridden by the reviewer",
    "Failure-selected inspection of training data: the exported cases do not estimate a failure rate.",
]
P.write_report(OUTPUT_ROOT, CASES, lines, SEL)
update_manifest(summary=lines, coverage_file="coverage.json")
print("\n".join(lines))
print(f"\nOpen {OUTPUT_ROOT}/report.html (download the folder, or use 9b) to browse cases and play the videos.")

In [ ]:
# @title 9b. (Optional) Make a downloadable archive of the report and cases
INCLUDE = ["report.html", "summary.csv", "coverage.json", "run_manifest.json", "selection.json",
           "selected_failure_videos.json", "key_mapping.json", "cases", "failure_videos", "reviews"]
size = 0
for p in INCLUDE:
    full = os.path.join(OUTPUT_ROOT, p)
    if os.path.isdir(full):
        size += sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(full) for f in fs)
    elif os.path.exists(full):
        size += os.path.getsize(full)
print(f"Archive would be about {size / 1e9:.2f} GB (videos included).")
MAKE_ZIP = False  # @param {type:"boolean"}
if MAKE_ZIP:
    stage = os.path.join(LOCAL_ROOT, "zip_stage", RUN_ID)
    shutil.rmtree(stage, ignore_errors=True)
    for p in INCLUDE:
        full = os.path.join(OUTPUT_ROOT, p)
        if os.path.isdir(full):
            shutil.copytree(full, os.path.join(stage, p))
        elif os.path.exists(full):
            os.makedirs(stage, exist_ok=True)
            shutil.copy(full, stage)
    z = shutil.make_archive(os.path.join(OUTPUT_ROOT, "..", RUN_ID + "_review"), "zip", os.path.dirname(stage), RUN_ID)
    print("Saved", z)

## Verification

In [ ]:
# @title V1. Checks (no GPU): selection, ids, prompts, media lookup and safe extraction, resume, scoring, frames, counting
r = subprocess.run([sys.executable, "/content/vsi590k/vsi590k_selftest.py"] +
                   ([ANNOTATION_PATHS[0]] if "ANNOTATION_PATHS" in globals() else []), capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-2000:])

In [ ]:
# @title (Optional) Unmount Google Drive after checking the outputs
from google.colab import drive
drive.flush_and_unmount()